# FORGE v19.5 Training-Prior Local/API Scorecard Notebook — ARC-AGI-3

Source lineage: **last strongest local base = `sigilagisilver.ipynb` / FORGE v19.5 inline gameplay lister**.

This notebook is built for **local development + ARC API scorecard runs**, not only Kaggle rerun submission.

## Modes

| `RUN_TARGET` | Meaning | Scorecard visibility |
|---|---|---|
| `offline` | Uses bundled local `environment_files` only | local JSON/object only |
| `api` | Uses ARC API online mode | visible ARC scorecard URL after close |
| `api_competition` | Uses ARC API competition mode | stricter, no inflight scorecard reads |

Default is `api` because you said you want to see the scorecard.

## Secret handling

The notebook **does not hardcode your API key**. Set it as `ARC_API_KEY` in your shell, notebook secrets, or paste it into the prompted field when running locally.


## Added in this build

This version consumes `gaming_strategy_dataset.tar.gz` as an **offline agent-training prior**:

```text
strategy dataset → action/role statistics → strategy_prior.json → agent exploration bias + opening curriculum
```

The dataset is not treated as direct button imitation. It is converted into ARC-safe priors over `RESET` / `ACTION1..ACTION7`, with raw console-control terms rejected by the dataset's own guardrails.


In [ ]:
# ============================================================
# 0) Config
# ============================================================
import os, sys, json, time, shutil, zipfile, subprocess, pathlib, textwrap, logging, traceback, compileall
from pathlib import Path

# Choose: "api", "api_competition", or "offline"
RUN_TARGET = os.environ.get("RUN_TARGET", "api").strip().lower()

# Empty means all games returned by the selected mode. Example: "ls20,g50t,ft09"
GAME_FILTER = os.environ.get("GAME_FILTER", "").strip()
MAX_GAMES = int(os.environ.get("MAX_GAMES", "0"))  # 0 = no cap

# Verbose gameplay lister controls from FORGE v19.5.
os.environ.setdefault("FORGE_GAMEPLAY_LOG", "1")
os.environ.setdefault("FORGE_GAMEPLAY_TRACE", "2")  # 0 quiet, 1 normal, 2 detailed, 3 heavy
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("DEBUG", "True")

# Runtime root. Keeps everything local to the notebook directory.
ROOT = Path.cwd().resolve()
LOCAL_ROOT = ROOT / "arc_agi3_local"
REPO_ROOT = LOCAL_ROOT / "ARC-AGI-3-Agents"
ENV_DIR = LOCAL_ROOT / "environment_files"
WHEEL_DIR = LOCAL_ROOT / "arc_agi_3_wheels"
RECORDINGS_DIR = ROOT / "recordings"
LOG_DIR = ROOT / "run_logs"
RECORDINGS_DIR.mkdir(exist_ok=True, parents=True)
LOG_DIR.mkdir(exist_ok=True, parents=True)

# ZIP candidates: current directory, /mnt/data, Kaggle input mirror.
ZIP_CANDIDATES = [
    ROOT / "arc-prize-2026-arc-agi-3.zip",
    Path("/mnt/data/arc-prize-2026-arc-agi-3.zip"),
    Path("/kaggle/input/arc-prize-2026-arc-agi-3.zip"),
    Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3.zip"),
]

print("[CONFIG] RUN_TARGET=", RUN_TARGET)
print("[CONFIG] ROOT=", ROOT)
print("[CONFIG] GAME_FILTER=", GAME_FILTER or "<all>")
print("[CONFIG] MAX_GAMES=", MAX_GAMES or "<no cap>")
# Offline strategy-prior training corpus.
USE_AGENT_TRAINING = os.environ.get("USE_AGENT_TRAINING", "1").strip() not in ("0", "false", "False", "no", "NO")
TRAINING_TAR_CANDIDATES = [
    ROOT / "gaming_strategy_dataset.tar.gz",
    Path("/mnt/data/gaming_strategy_dataset.tar.gz"),
    Path("/kaggle/input/gaming-strategy-dataset/gaming_strategy_dataset.tar.gz"),
    Path("/kaggle/input/nintendo-gaming-strategy-dataset/gaming_strategy_dataset.tar.gz"),
]
TRAINING_ROOT = ROOT / "agent_training"
TRAINING_EXTRACT_DIR = TRAINING_ROOT / "gaming_strategy_dataset"
STRATEGY_PRIOR_PATH = TRAINING_ROOT / "strategy_prior.json"
os.environ.setdefault("FORGE_USE_STRATEGY_PRIOR", "1" if USE_AGENT_TRAINING else "0")
os.environ.setdefault("FORGE_STRATEGY_PRIOR_STRENGTH", "0.35")

print("[CONFIG] USE_AGENT_TRAINING=", USE_AGENT_TRAINING)
print("[CONFIG] STRATEGY_PRIOR_PATH=", STRATEGY_PRIOR_PATH)


In [ ]:
# ============================================================
# 1) Extract local ARC-AGI-3 bundle if present
# ============================================================
def find_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

zip_path = find_existing(ZIP_CANDIDATES)
if zip_path and not (REPO_ROOT / "main.py").exists():
    print(f"[SETUP] Extracting {zip_path} -> {LOCAL_ROOT}")
    LOCAL_ROOT.mkdir(exist_ok=True, parents=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(LOCAL_ROOT)
else:
    print("[SETUP] Existing local repo found or no zip extraction needed")

# Some archives extract top-level dirs into LOCAL_ROOT; normalize expected paths.
if not REPO_ROOT.exists() and (LOCAL_ROOT / "ARC-AGI-3-Agents").exists():
    REPO_ROOT = LOCAL_ROOT / "ARC-AGI-3-Agents"

# Kaggle fallback paths.
KAGGLE_WHEEL_DIR = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
KAGGLE_ENV_DIR = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files")
if not WHEEL_DIR.exists() and KAGGLE_WHEEL_DIR.exists():
    WHEEL_DIR = KAGGLE_WHEEL_DIR
if not ENV_DIR.exists() and KAGGLE_ENV_DIR.exists():
    ENV_DIR = KAGGLE_ENV_DIR

print("[SETUP] REPO_ROOT=", REPO_ROOT, "exists=", REPO_ROOT.exists())
print("[SETUP] WHEEL_DIR=", WHEEL_DIR, "exists=", WHEEL_DIR.exists())
print("[SETUP] ENV_DIR=", ENV_DIR, "exists=", ENV_DIR.exists())


In [ ]:
# ============================================================
# 2.5) Extract gaming_strategy_dataset.tar.gz and build agent-training prior
# ============================================================
import tarfile, math
from collections import Counter, defaultdict

TRAINING_ROOT.mkdir(parents=True, exist_ok=True)

def _safe_extract_tar_gz(tar_path: Path, dest: Path):
    """Extract tar safely: reject absolute paths and path traversal."""
    dest = dest.resolve()
    if dest.exists() and any(dest.iterdir()):
        print(f"[TRAINING] Existing extraction found: {dest}")
        return
    dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:gz") as tf:
        members = []
        for m in tf.getmembers():
            target = (dest / m.name).resolve()
            if not str(target).startswith(str(dest)):
                raise RuntimeError(f"Blocked unsafe tar member: {m.name}")
            members.append(m)
        tf.extractall(dest, members=members)
    print(f"[TRAINING] Extracted {tar_path} -> {dest}")

def _iter_jsonl(path: Path):
    if not path.exists():
        return
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception:
                continue

def _normalize_counter(counter):
    total = float(sum(max(v, 0.0) for v in counter.values())) or 1.0
    return {k: float(v) / total for k, v in sorted(counter.items())}

def build_strategy_prior(dataset_dir: Path, out_path: Path):
    """
    Build a compact ARC-safe prior from the uploaded strategy corpus.

    This is not supervised frame/action training. The corpus was text-derived, so the
    safe conversion is: strategy text -> semantic action priors -> exploration bias.
    """
    hints_path = dataset_dir / "arc_agi3_control_hints.jsonl"
    facts_path = dataset_dir / "strategy_facts.jsonl"
    quality_path = dataset_dir / "quality_report.json"
    ontology_path = dataset_dir / "arc_agi3_action_ontology.json"

    action_counts = Counter()
    action_conf = Counter()
    role_counts = Counter()
    role_action_conf = defaultdict(Counter)
    fact_type_counts = Counter()
    doc_counts = Counter()

    for o in _iter_jsonl(hints_path):
        if not o.get("arc_only", True):
            continue
        act = o.get("arc_action")
        role = o.get("semantic_role", "unknown")
        conf = float(o.get("confidence", 0.0) or 0.0)
        if act:
            action_counts[act] += 1
            action_conf[act] += conf
            role_action_conf[role][act] += conf
        role_counts[role] += 1

    for o in _iter_jsonl(facts_path):
        for ft in o.get("fact_types", []) or []:
            fact_type_counts[ft] += 1
        doc = o.get("document_id") or "unknown"
        doc_counts[doc] += 1

    quality = {}
    ontology = {}
    if quality_path.exists():
        quality = json.loads(quality_path.read_text(encoding="utf-8"))
    if ontology_path.exists():
        ontology = json.loads(ontology_path.read_text(encoding="utf-8"))

    allowed = set(ontology.get("allowed_executable_controls", ["RESET", "ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION6", "ACTION7"]))
    allowed_no_reset = [a for a in ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION6", "ACTION7"] if a in allowed]

    probs = _normalize_counter({a: action_conf[a] for a in allowed_no_reset})
    # Convert probabilities into small, mean-centered logit biases.
    mean_prob = 1.0 / max(len(allowed_no_reset), 1)
    log_bias = {}
    for a in allowed_no_reset:
        p = max(probs.get(a, 1e-9), 1e-9)
        log_bias[a] = max(-0.75, min(0.75, math.log(p / mean_prob)))

    # ACTION1..5 are direct action logits. ACTION6 click is separate. ACTION7 undo is kept as a repair prior.
    action_bias_vector = [float(log_bias.get(f"ACTION{i}", 0.0)) for i in range(1, 6)]
    click_bias = float(log_bias.get("ACTION6", 0.0))
    undo_bias = float(log_bias.get("ACTION7", 0.0))

    # Opening curriculum: repeat high-confidence non-undo, non-click actions first.
    ranked_open = [a for a, _ in action_conf.most_common() if a in {"ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5"}]
    if not ranked_open:
        ranked_open = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5"]
    # Avoid a one-direction rut: interleave the learned top order with coverage.
    coverage = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5"]
    opening = []
    for a in ranked_open + coverage + ranked_open[:3]:
        if a not in opening or len(opening) < 8:
            opening.append(a)
    opening_action_ids = [int(a.replace("ACTION", "")) for a in opening[:10]]

    prior = {
        "record_type": "forge_strategy_prior_v1",
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "source_dataset_dir": str(dataset_dir),
        "quality_report": quality,
        "allowed_executable_controls": sorted(allowed),
        "action_counts": dict(action_counts),
        "action_confidence_sums": {k: float(v) for k, v in action_conf.items()},
        "action_probabilities": probs,
        "action_logit_bias": log_bias,
        "action_bias_vector_ACTION1_to_ACTION5": action_bias_vector,
        "click_ACTION6_bias": click_bias,
        "undo_ACTION7_bias": undo_bias,
        "opening_action_ids": opening_action_ids,
        "semantic_role_counts": dict(role_counts),
        "fact_type_counts": dict(fact_type_counts),
        "role_action_confidence": {role: dict(cnt) for role, cnt in role_action_conf.items()},
        "top_documents": dict(doc_counts.most_common(20)),
        "training_policy": {
            "safe_mode": True,
            "raw_console_controls_executable": False,
            "use_as": "exploration_prior_and_opening_curriculum",
            "not_use_as": "direct_console_button_imitation",
        },
    }
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(prior, indent=2, sort_keys=True), encoding="utf-8")
    return prior

if USE_AGENT_TRAINING:
    training_tar = find_existing(TRAINING_TAR_CANDIDATES)
    if not training_tar:
        print("[TRAINING][WARN] gaming_strategy_dataset.tar.gz not found. Agent will run without training prior.")
        os.environ["FORGE_USE_STRATEGY_PRIOR"] = "0"
    else:
        _safe_extract_tar_gz(training_tar, TRAINING_EXTRACT_DIR)
        # Tar contains a top-level gaming_strategy_dataset directory.
        dataset_dir = TRAINING_EXTRACT_DIR / "gaming_strategy_dataset"
        if not dataset_dir.exists():
            dataset_dir = TRAINING_EXTRACT_DIR
        prior = build_strategy_prior(dataset_dir, STRATEGY_PRIOR_PATH)
        os.environ["FORGE_STRATEGY_PRIOR_PATH"] = str(STRATEGY_PRIOR_PATH)
        print("[TRAINING] Built strategy prior:", STRATEGY_PRIOR_PATH)
        print("[TRAINING] action probabilities:", prior["action_probabilities"])
        print("[TRAINING] opening_action_ids:", prior["opening_action_ids"])
        print("[TRAINING] control guard passed:", prior.get("quality_report", {}).get("arc_only_control_guard_passed"))
else:
    os.environ["FORGE_USE_STRATEGY_PRIOR"] = "0"
    print("[TRAINING] Disabled by USE_AGENT_TRAINING=0")


In [ ]:
# ============================================================
# 2) Install ARC packages from local wheels when needed
# ============================================================
def have_module(name: str) -> bool:
    try:
        __import__(name)
        return True
    except Exception:
        return False

if not have_module("arc_agi") or not have_module("arcengine"):
    if WHEEL_DIR.exists():
        print("[SETUP] Installing arc-agi/python-dotenv from wheels:", WHEEL_DIR)
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(WHEEL_DIR),
            "arc-agi", "python-dotenv"
        ])
    else:
        print("[SETUP][WARN] Local wheels not found. Trying normal pip install; this needs internet.")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "arc-agi", "python-dotenv"])
else:
    print("[SETUP] arc_agi + arcengine already importable")

import arc_agi, arcengine
print("[SETUP] arc_agi imported from", getattr(arc_agi, "__file__", "<unknown>"))
print("[SETUP] arcengine imported from", getattr(arcengine, "__file__", "<unknown>"))


## 3) Write the FORGE v19.5 agent

This cell writes the full last-highest local base to `my_agent.py`. It is intentionally large.


In [ ]:
%%writefile my_agent.py
# =====================================================================
# FORGE v19.5 — inline gameplay lister + defined movement graft
#
# Fixes applied on top of v18:
#
# FIX 1: _visited_hashes was never initialized in __init__ — reward
#         signal was broken: always gave +1.5 for ANY hash change,
#         never penalizing loops. Now properly tracks and deduplicates.
#
# FIX 2: CLTI frame extraction used get_pixels() which is inconsistent
#         with _raw() (which reads frame[-1] from perform_action).
#         Now uses perform_action result frames throughout, so injected
#         expert demos have correct state representations.
#
# FIX 3: BFS hidden retry used 3 RESET calls instead of 2, landing
#         in a different initial state than the first pass scan,
#         causing the retry to search from a mismatched baseline.
#
# FIX 4: Epsilon always reset to 0.15 on level change even when BFS
#         already solved the level. Now only resets if BFS failed,
#         preserving learned exploration for CNN fallback.
#
# v19.4 definition:
# - Deterministic hybrid ARC-AGI-3 agent: direct game introspection first, learned CNN fallback second
# - Movement data = action-conditioned pixel delta + hidden scalar trigger/counter delta + level-to-level transfer
# - Safety rule: never compress or mutate a validated BFS solution unless replay validation passes
#
# Movement-data graft retained:
# - v16/v20 trigger-aware movement fallbacks
# - click-hit retention without effect dedup
# - stride-1 neighbor probing around clicked sprites
# - offset + multiplier transfer replay
# - final BFS click data emission fix
# =====================================================================
import copy
import glob
import heapq
import hashlib
import importlib.util
import logging
import os
import json
import random
import re
import time
import traceback
from collections import deque
from itertools import permutations

import numpy as np

# ---------------------------------------------------------------------
# Termux/Android-safe optional torch gate.
# Full Torch is used on Kaggle/Linux. On Termux, torch.nn may be absent;
# then we keep deterministic BFS + heuristic + strategy-prior fallback.
# ---------------------------------------------------------------------
FORGE_TORCH_AVAILABLE = False
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    _ = nn.Module
    FORGE_TORCH_AVAILABLE = True
except Exception as _forge_torch_error:
    FORGE_TORCH_AVAILABLE = False

    class _FakeTensor:
        device = "cpu"
        dtype = "float32"
        def __init__(self, *a, **k): pass
        def to(self, *a, **k): return self
        def float(self): return self
        def long(self): return self
        def clamp(self, *a, **k): return self
        def unsqueeze(self, *a, **k): return self
        def squeeze(self, *a, **k): return self
        def reshape(self, *a, **k): return self
        def view(self, *a, **k): return self
        def mean(self, *a, **k): return self
        def max(self, *a, **k): return (self, self)
        def sum(self, *a, **k): return 1.0
        def item(self): return 0.0
        def __getitem__(self, k): return self
        def __setitem__(self, k, v): pass
        def __add__(self, o): return self
        def __radd__(self, o): return self
        def __sub__(self, o): return self
        def __mul__(self, o): return self
        def __rmul__(self, o): return self
        def __truediv__(self, o): return self
        def backward(self): pass
        def scatter_(self, *a, **k): return self

    class _NullModule:
        def __init__(self, *a, **k): pass
        def to(self, *a, **k): return self
        def parameters(self): return []
        def state_dict(self): return {}
        def load_state_dict(self, *a, **k): return self
        def __call__(self, *a, **k): return _FakeTensor()

    class _NN:
        Module = _NullModule
        Linear = _NullModule
        Conv2d = _NullModule
        MaxPool2d = _NullModule
        Dropout = _NullModule
        AdaptiveAvgPool2d = _NullModule
        Flatten = _NullModule
        ReLU = _NullModule
        Sequential = _NullModule

    class _F:
        @staticmethod
        def relu(x): return x
        @staticmethod
        def softmax(x, *a, **k): return x
        @staticmethod
        def one_hot(x, *a, **k): return x
        @staticmethod
        def binary_cross_entropy_with_logits(*a, **k): return _FakeTensor()

    class _Optim:
        class Adam:
            def __init__(self, *a, **k): pass
            def zero_grad(self): pass
            def step(self): pass

    class _NoGrad:
        def __enter__(self): return self
        def __exit__(self, *a): return False

    class _FakeCuda:
        @staticmethod
        def is_available(): return False

    class _FakeMPS:
        @staticmethod
        def is_available(): return False

    class _FakeBackends:
        mps = _FakeMPS()

    class _Torch:
        float32 = "float32"
        long = "long"
        cuda = _FakeCuda()
        backends = _FakeBackends()
        @staticmethod
        def manual_seed(*a, **k): return None
        @staticmethod
        def device(x): return "cpu"
        @staticmethod
        def zeros(*a, **k): return _FakeTensor()
        @staticmethod
        def ones(*a, **k): return _FakeTensor()
        @staticmethod
        def tensor(*a, **k): return _FakeTensor()
        @staticmethod
        def from_numpy(*a, **k): return _FakeTensor()
        @staticmethod
        def stack(*a, **k): return _FakeTensor()
        @staticmethod
        def cat(*a, **k): return _FakeTensor()
        @staticmethod
        def sigmoid(x): return x
        @staticmethod
        def log(x): return x
        @staticmethod
        def zeros_like(x, *a, **k): return _FakeTensor()
        @staticmethod
        def ones_like(x, *a, **k): return _FakeTensor()
        @staticmethod
        def full_like(x, *a, **k): return _FakeTensor()
        @staticmethod
        def bmm(*a, **k): return _FakeTensor()
        @staticmethod
        def load(*a, **k): return {}
        @staticmethod
        def no_grad(): return _NoGrad()

    torch = _Torch()
    nn = _NN()
    F = _F()
    optim = _Optim()
    print("[FORGE][WARN] torch.nn unavailable; using Termux torchless fallback")

from agents.agent import Agent
from arcengine import FrameData, GameAction, GameState, ActionInput

logger = logging.getLogger(__name__)

# Competition-grade deterministic defaults. These avoid run-to-run variance in
# fallback probes while keeping Kaggle runtime self-contained.
FORGE_VERSION = "19.5-inline-gameplay-lister"
random.seed(918)
np.random.seed(918)
torch.manual_seed(918)


# ==================== BFS SOLVER ====================
def _fast_deepcopy(game):
    """Deepcopy game object, skipping the camera (rendering-only, never mutates)."""
    camera = game._camera
    game._camera = None
    g = copy.deepcopy(game)
    game._camera = camera
    g._camera = camera
    return g

class BFSSolver:
    """Offline BFS solver using direct game class instantiation."""

    def __init__(self, game_path, game_class_name, scan_timeout=3, bfs_timeout=120):
        self.game_path = game_path
        self.class_name = game_class_name
        self.scan_timeout = scan_timeout
        self.bfs_timeout = bfs_timeout
        self.game_cls = None
        self.solutions = {}  # level_idx → action list
        self.timed_out_levels = set()

    def load(self):
        """Load the game class from source."""
        try:
            spec = importlib.util.spec_from_file_location('game_mod', self.game_path)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            self.game_cls = getattr(mod, self.class_name)
            return True
        except Exception as e:
            logger.warning(f"BFS: Failed to load game class: {e}")
            return False

    def _save_state(self, game):
        return copy.deepcopy(game.__dict__)

    def _restore_state(self, base_game, state_dict):
        g = copy.deepcopy(base_game)
        g.__dict__.update(copy.deepcopy(state_dict))
        return g

    def _perform_and_drain(self, game, ai, max_drain=5, drain=True):
        try:
            r = game.perform_action(ai, raw=True)
        except Exception as e:
            logger.warning(f"BFS drain: initial perform_action failed: {e}")
            raise
        if not drain or not r.frame:
            return r
    
        prev_frame = np.array(r.frame[-1])
        for _ in range(max_drain):
            try:
                r2 = game.perform_action(ActionInput(id=GameAction.ACTION1), raw=True)
            except:
                break
            if not r2.frame:
                break
            curr_frame = np.array(r2.frame[-1])
            if np.array_equal(curr_frame, prev_frame):
                break
            r = r2
            prev_frame = curr_frame
        return r

    def _analyse_demo(self, frames_and_actions):
        """Analyse a demonstration (sequence of frame, action pairs) to extract:
        - Which colors are player-controlled (move in response to actions)
        - Which colors are passive targets (stationary until win)
        - What the win condition looks like structurally
        
        Returns a demo_model dict with this information.
        """
        if len(frames_and_actions) < 2:
            return None
        
        bg = int(np.bincount(
            frames_and_actions[0][0].flatten(), minlength=16).argmax())
        
        # Action direction vectors
        action_dirs = {1: (0,-1), 2: (0,1), 3: (-1,0), 4: (1,0)}
        
        def get_centroids(frame):
            result = {}
            for c in range(16):
                if c == bg: continue
                mask = (frame == c)
                n = int(np.sum(mask))
                if n < 4: continue
                ys, xs = np.where(mask)
                result[c] = (float(np.mean(xs)), float(np.mean(ys)), n)
            return result
        
        # Track per-color movement correlation with action direction
        # player-controlled colors move in the action direction
        color_action_corr = {}  # color -> list of (expected_dx, actual_dx, expected_dy, actual_dy)
        color_movement = {}     # color -> total movement across all steps
        
        prev_frame, _ = frames_and_actions[0]
        prev_centroids = get_centroids(prev_frame)
        
        for frame, action in frames_and_actions[1:]:
            curr_centroids = get_centroids(frame)
            adx, ady = action_dirs.get(action, (0, 0))
            
            for c in prev_centroids:
                if c not in curr_centroids:
                    continue
                actual_dx = curr_centroids[c][0] - prev_centroids[c][0]
                actual_dy = curr_centroids[c][1] - prev_centroids[c][1]
                movement = abs(actual_dx) + abs(actual_dy)
                
                if c not in color_action_corr:
                    color_action_corr[c] = []
                    color_movement[c] = 0
                color_movement[c] += movement
                
                # Does this color move in the action direction?
                if movement > 1:
                    if adx != 0:
                        corr = np.sign(actual_dx) == np.sign(adx)
                    elif ady != 0:
                        corr = np.sign(actual_dy) == np.sign(ady)
                    else:
                        corr = False
                    color_action_corr[c].append(corr)
            
            prev_frame = frame
            prev_centroids = curr_centroids
        
        # Track pixel count stability per color
        # Player colors maintain consistent pixel counts
        # Target colors that get overlapped show sudden pixel count changes at win step
        color_pixel_counts = {}  # color -> list of pixel counts across frames
        for frame, action in frames_and_actions:
            c_counts = {}
            for c in range(16):
                if c == bg: continue
                n = int(np.sum(frame == c))
                if n >= 4:
                    c_counts[c] = n
            for c, n in c_counts.items():
                if c not in color_pixel_counts:
                    color_pixel_counts[c] = []
                color_pixel_counts[c].append(n)
    
        player_colors = set()
        passive_colors = set()
        for c, corrs in color_action_corr.items():
            total_movement = color_movement.get(c, 0)
            
            # Check pixel count stability
            counts = color_pixel_counts.get(c, [])
            if len(counts) >= 2:
                count_variance = max(counts) - min(counts)
                # High variance in pixel count = color appears/disappears = target being overlapped
                count_stable = count_variance < max(counts) * 0.3
            else:
                count_stable = True
    
            if not corrs:
                if total_movement < 1:
                    passive_colors.add(c)
                continue
            corr_rate = sum(corrs) / len(corrs)
            if corr_rate > 0.5 and total_movement > 5 and count_stable:
                player_colors.add(c)
            elif corr_rate < 0.3 or not count_stable:
                passive_colors.add(c)
        
        # Win frame analysis
        win_frame = frames_and_actions[-1][0]
        init_frame = frames_and_actions[0][0]
        win_centroids = get_centroids(win_frame)
        init_centroids = get_centroids(init_frame)
        
        # What changed at the win step vs second-to-last step?
        pre_win_frame = frames_and_actions[-2][0]
        pre_win_centroids = get_centroids(pre_win_frame)
        
        win_changes = {}  # color -> (pre_win_pos, win_pos)
        for c in pre_win_centroids:
            if c not in win_centroids:
                continue
            dx = abs(win_centroids[c][0] - pre_win_centroids[c][0])
            dy = abs(win_centroids[c][1] - pre_win_centroids[c][1])
            if dx + dy > 2:
                win_changes[c] = (
                    (pre_win_centroids[c][0], pre_win_centroids[c][1]),
                    (win_centroids[c][0], win_centroids[c][1])
                )
        
       # Win conditions: which player colors moved TOWARD passive colors at the win step?
        # Compare pre-win distance vs post-win distance for each (player, passive) pair
        win_conditions = []
        for pc in player_colors:
            if pc not in win_centroids or pc not in pre_win_centroids:
                continue
            for tc in passive_colors:
                if tc not in win_centroids or tc not in pre_win_centroids:
                    continue
                # Distance before and after win step
                pre_dist = (abs(pre_win_centroids[pc][0] - pre_win_centroids[tc][0]) +
                           abs(pre_win_centroids[pc][1] - pre_win_centroids[tc][1]))
                post_dist = (abs(win_centroids[pc][0] - win_centroids[tc][0]) +
                            abs(win_centroids[pc][1] - win_centroids[tc][1]))
                # Player color moved toward passive color at win step
                if post_dist < pre_dist and post_dist < 15:
                    win_conditions.append((pc, tc))
        
        # Pixel-level win signature: what transformation happened?
        changed_mask = init_frame != win_frame
        n_changed = int(np.sum(changed_mask))
        
        return {
            'player_colors': player_colors,
            'passive_colors': passive_colors,
            'win_conditions': win_conditions,  # (player_color, target_color) pairs
            'win_centroids': win_centroids,
            'init_centroids': init_centroids,
            'bg': bg,
            'n_changed': n_changed,
            'win_frame': win_frame,
            'init_frame': init_frame,
        }

    def _build_goal_heuristic(self, f_init, f_prev_win, demo_model=None):
        """Build A* heuristic using game-state introspection.
        
        Scans game object for indicator sprites (any dict->list->sprite
        with is_visible property) and counts unsatisfied conditions.
        Falls back to uniform cost if no indicators found.
        General: works for any game using the indicator pattern.
        """
        def introspection_heuristic(f, game=None):
            if game is None:
                return 0
            try:
                total, satisfied = 0, 0
                for attr_val in game.__dict__.values():
                    if not isinstance(attr_val, dict):
                        continue
                    for v in attr_val.values():
                        if not isinstance(v, list):
                            continue
                        for item in v:
                            if hasattr(item, 'is_visible') and hasattr(item, 'pixels'):
                                total += 1
                                if item.is_visible:
                                    satisfied += 1
                if total == 0:
                    return 0
                return total - satisfied
            except:
                return 0

        # Validate signal exists on a fresh game instance
        if self.game_cls:
            try:
                test = self.game_cls()
                test.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                test.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                h = introspection_heuristic(None, test)
                if h > 0:
                    logger.info(f"BFS heuristic: introspection found {h} indicators")
                    return introspection_heuristic
            except:
                pass

        logger.info(f"BFS heuristic: no indicators found, uniform cost")
        return lambda f, game=None: 0
     
    def _state_hash(self, g, frame, hidden_fields=None, transient_fields=None):
        """Hash visible frame plus selected scalar state.

        If hidden_fields is None, preserve the v19 broad scalar hash.
        If hidden_fields is supplied, use only those trigger/counter fields.
        This lets the v16/v20 movement fallbacks avoid clock/counter blowups.
        """
        fh = hashlib.md5(frame.tobytes()).hexdigest()[:16]
        ignore = {'_action_count', '_full_reset', '_action_complete', '_debug', '_seed'}
        if transient_fields:
            ignore.update(transient_fields)

        extras = []
        field_filter = set(hidden_fields) if hidden_fields is not None else None
        for k, v in g.__dict__.items():
            if k.startswith('__') or k in ignore:
                continue
            if field_filter is not None and k not in field_filter:
                continue
            if isinstance(v, (int, float, bool)):
                extras.append(f"{k}={v}")
            elif isinstance(v, (set, frozenset)) and len(v) < 50:
                extras.append(f"{k}={sorted(str(i) for i in v)}")
        if extras:
            eh = hashlib.md5("|".join(sorted(extras)).encode()).hexdigest()[:12]
            return fh + "|" + eh
        return fh

    def _extract_win_field(self):
        """Extract likely win-condition counter/flag from game source."""
        try:
            source = open(self.game_path, encoding="utf-8", errors="ignore").read()
            lines = source.split('\n')
            for i, line in enumerate(lines):
                if 'self.next_level()' in line:
                    for j in range(i - 1, max(0, i - 10), -1):
                        s = lines[j].strip()
                        if s.startswith('if ') or s.startswith('elif '):
                            m = re.search(r'self\.(\w+)', s)
                            if m:
                                return m.group(1)
                    break
        except:
            pass
        return None

    def _probe_hidden_fields(self, game, actions):
        """Dynamic state probing with win-field awareness.

        Keeps fields useful for trigger/counter movement search while filtering
        obvious engine book-keeping fields.
        """
        if not actions:
            return []
        initial = {}
        for k, v in game.__dict__.items():
            if isinstance(v, (int, float, bool)) and not k.startswith('__'):
                initial[k] = v

        changing_fields = set()
        win_field = self._extract_win_field()
        if win_field and win_field in initial:
            changing_fields.add(win_field)

        try:
            frame0 = np.array(game.get_pixels(0, 0, 64, 64))
        except:
            frame0 = None

        for act_id, data in actions[:12]:
            g = copy.deepcopy(game)
            try:
                ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                g.perform_action(ai, raw=True)
            except:
                continue
            pixels_changed = False
            if frame0 is not None:
                try:
                    f = np.array(g.get_pixels(0, 0, 64, 64))
                    pixels_changed = bool(np.sum(frame0 != f) > 0)
                except:
                    pixels_changed = False
            for k, v in g.__dict__.items():
                if isinstance(v, (int, float, bool)) and not k.startswith('__'):
                    if k in initial and v != initial[k]:
                        if k not in ('_action_count', '_full_reset', '_action_complete'):
                            # Keep hidden trigger fields and explicit win counters.
                            if (not pixels_changed) or k == win_field or not k.startswith('_'):
                                changing_fields.add(k)

        hidden = []
        for f in changing_fields:
            if f.startswith('_') and f not in ('_current_level_index', '_score', win_field):
                continue
            hidden.append(f)
        return sorted(hidden)

    def _detect_transient_fields(self, game, actions):
        """Detect scalar fields that change on every action (e.g. budget counters,
        monotonic clocks). These add no state-distinguishing value to the hash and
        cause state space explosion if included."""
        if not actions:
            return set()
        initial = {k: v for k, v in game.__dict__.items()
                   if isinstance(v, (int, float, bool)) and not k.startswith('__')
                   and k not in ('_action_count', '_full_reset', '_action_complete')}
        # Track how many sampled actions changed each field
        changed_count = {k: 0 for k in initial}
        n_sampled = 0
        for act_id, data in actions[:min(12, len(actions))]:
            g = copy.deepcopy(game)
            try:
                ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                g.perform_action(ai, raw=True)
            except:
                continue
            n_sampled += 1
            for k in initial:
                if getattr(g, k, initial[k]) != initial[k]:
                    changed_count[k] += 1
        # Also sample click actions so click-triggered transients are detected
        if hasattr(game, '_get_valid_actions'):
            try:
                for va in game._get_valid_actions()[:4]:
                    g = copy.deepcopy(game)
                    try:
                        g.perform_action(va, raw=True)
                    except:
                        continue
                    n_sampled += 1
                    for k in initial:
                        if getattr(g, k, initial[k]) != initial[k]:
                            changed_count[k] += 1
            except:
                pass            
        if n_sampled == 0:
            return set()
        # A field is transient if it changed in every sampled action
        # Exclude monotonic counters (always decrease/increase) but keep boolean flags
        # Boolean flags encode meaningful state (e.g. which object is selected)
        transient = set()
        for k, cnt in changed_count.items():
            if cnt != n_sampled:
                continue
            v = initial[k]
            if isinstance(v, bool):
                continue  # boolean flags are meaningful state, never transient
            transient.add(k)
        if transient:
            logger.info(f"BFS: detected transient fields (excluded from hash): {transient}")
        return transient
    
    def _build_goal_heuristic(self, f_init, f_prev_win, demo_model=None):
    
        def count_indicators(game):
            try:
                total, satisfied = 0, 0
                for av in game.__dict__.values():
                    if not isinstance(av, dict): continue
                    for v in av.values():
                        if not isinstance(v, list): continue
                        for item in v:
                            if hasattr(item, 'is_visible') and hasattr(item, 'pixels'):
                                total += 1
                                if item.is_visible: satisfied += 1
                return total, satisfied
            except:
                return 0, 0
    
        # Cache selectable actions at heuristic build time, not per node
        cached_selectable_actions = []
        if self.game_cls:
            try:
                test = self.game_cls()
                test.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                test.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                if 6 in test._available_actions and hasattr(test, '_get_valid_actions'):
                    f0 = np.array(test.perform_action(
                        ActionInput(id=GameAction.ACTION1), raw=True).frame[-1])
                    bg = int(np.bincount(f0.flatten(), minlength=16).argmax())
                    # detect once here, store action inputs only
                    seen = set()
                    for va in test._get_valid_actions():
                        act_id = va.id._value_ if hasattr(va.id, '_value_') else int(va.id)
                        if act_id == 6:
                            cached_selectable_actions.append(va)
            except:
                pass
    
        def introspection_heuristic(f, game=None):
            if game is None:
                return 0
            try:
                total, satisfied = count_indicators(game)
                if total == 0:
                    return 0
                base_cost = total - satisfied
                # Use pre-cached selectable actions — no deepcopy detection per node
                extra_cost = 0
                for va in cached_selectable_actions:
                    gc = copy.deepcopy(game)
                    try:
                        gc.perform_action(va, raw=True)
                        t, s = count_indicators(gc)
                        if t > 0:
                            extra_cost += (t - s)
                    except:
                        pass
                return base_cost + extra_cost
            except:
                return 0
    
        # Validate
        if self.game_cls:
            try:
                test = self.game_cls()
                test.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                test.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                total, _ = count_indicators(test)
                if total > 0:
                    logger.info(f"BFS heuristic: introspection found {total} indicators")
                    return introspection_heuristic
            except:
                pass
    
        logger.info(f"BFS heuristic: no indicators found, uniform cost")
        return lambda f, game=None: 0
        
    def _scan_actions(self, game, f0, bg):
        """Hybrid movement/action scan.

        Grafts v16/v20 movement data into v19:
        - directional/interact actions are probed for real movement
        - if no directional probe moves pixels, preserve all available base actions
        - click actions are retained by coordinate, not collapsed by effect hash
        - stride-1 neighbors around click hits catch odd-coordinate sprites
        """
        avail = list(getattr(game, '_available_actions', []) or [])
        actions = []
        seen = set()

        def _clean_data(data):
            if not data:
                return None
            try:
                d = dict(data)
            except:
                d = data
            return d

        def _key(act_id, data):
            if not data:
                return (act_id, None)
            if isinstance(data, dict):
                return (act_id, int(data.get('x', -1)), int(data.get('y', -1)))
            return (act_id, str(data))

        def _add(act_id, data=None):
            data = _clean_data(data)
            k = _key(act_id, data)
            if k not in seen:
                seen.add(k)
                actions.append((act_id, data))

        # Directional/interact actions: prefer effective movers, but preserve all
        # base actions if the game hides movement in internal state.
        directional_hits = 0
        for a in [a for a in avail if 1 <= a <= 5]:
            g = _fast_deepcopy(game)
            try:
                r = g.perform_action(ActionInput(id=GameAction.from_id(a)), raw=True)
                if r.frame and np.sum(f0 != np.array(r.frame[-1])) > 0:
                    _add(a, None)
                    directional_hits += 1
            except:
                pass
        if directional_hits == 0:
            for a in [a for a in avail if 1 <= a <= 5]:
                _add(a, None)

        # Click actions: use exact valid actions first, then pixel scan.
        if 6 in avail:
            t0 = time.time()
            hit_positions = []

            if hasattr(game, '_get_valid_actions'):
                try:
                    for ai_obj in game._get_valid_actions():
                        if time.time() - t0 > self.scan_timeout:
                            break
                        act_id = ai_obj.id._value_ if hasattr(ai_obj.id, '_value_') else int(ai_obj.id)
                        if act_id != 6:
                            continue
                        data = _clean_data(getattr(ai_obj, 'data', None)) or {}
                        x, y = int(data.get('x', -1)), int(data.get('y', -1))
                        g = _fast_deepcopy(game)
                        try:
                            r = g.perform_action(ai_obj, raw=True)
                            if r.frame and np.sum(f0 != np.array(r.frame[-1])) > 0:
                                if x >= 0 and y >= 0:
                                    data['game_id'] = 'bfs'
                                    _add(6, data)
                                    hit_positions.append((x, y))
                        except:
                            pass
                except:
                    pass

            # Raw stride-2 scan without effect dedup; this is the useful v16/v20
            # movement delta that preserved duplicate-looking but distinct clicks.
            for y in range(0, 64, 2):
                if time.time() - t0 > self.scan_timeout:
                    break
                for x in range(0, 64, 2):
                    if f0[y, x] == bg:
                        continue
                    data = {'x': x, 'y': y, 'game_id': 'bfs'}
                    if _key(6, data) in seen:
                        continue
                    g = _fast_deepcopy(game)
                    try:
                        r = g.perform_action(ActionInput(id=GameAction.ACTION6, data=data), raw=True)
                        if r.frame and np.sum(f0 != np.array(r.frame[-1])) > 0:
                            _add(6, data)
                            hit_positions.append((x, y))
                    except:
                        pass

            # Probe stride-1 neighbors of hits to catch odd-coordinate sprites.
            tried = {(x, y) for x, y in hit_positions}
            for hx, hy in list(hit_positions):
                if time.time() - t0 > self.scan_timeout * 1.5:
                    break
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nx, ny = hx + dx, hy + dy
                    if (nx, ny) in tried or not (0 <= nx < 64 and 0 <= ny < 64):
                        continue
                    tried.add((nx, ny))
                    if f0[ny, nx] == bg:
                        continue
                    data = {'x': nx, 'y': ny, 'game_id': 'bfs'}
                    g = _fast_deepcopy(game)
                    try:
                        r = g.perform_action(ActionInput(id=GameAction.ACTION6, data=data), raw=True)
                        if r.frame and np.sum(f0 != np.array(r.frame[-1])) > 0:
                            _add(6, data)
                    except:
                        pass

        return actions
        
    def _probe_mover_target_colors(self, game):
        """Classify colors as movers vs targets by running 20 random actions."""
        g = copy.deepcopy(game)
        avail = [a for a in game._available_actions if 1 <= a <= 4]
        if not avail:
            return set(), set()
        r0 = g.perform_action(ActionInput(id=GameAction.from_id(avail[0])), raw=True)
        if not r0.frame:
            return set(), set()
        f0 = np.array(r0.frame[-1])
        bg = int(np.bincount(f0.flatten(), minlength=16).argmax())
    
        def get_centroids(frame):
            result = {}
            for c in range(16):
                if c == bg: continue
                mask = (frame == c)
                n = int(np.sum(mask))
                if n < 2: continue
                ys, xs = np.where(mask)
                result[c] = (float(np.mean(xs)), float(np.mean(ys)))
            return result
    
        movement = {}
        prev_c = get_centroids(f0)
        for _ in range(20):
            act = random.choice(avail)
            try:
                r2 = g.perform_action(ActionInput(id=GameAction.from_id(act)), raw=True)
            except:
                break
            if not r2.frame:
                break
            curr_c = get_centroids(np.array(r2.frame[-1]))
            for c in prev_c:
                if c in curr_c:
                    movement[c] = movement.get(c, 0.0) + abs(curr_c[c][0] - prev_c[c][0]) + abs(curr_c[c][1] - prev_c[c][1])
            prev_c = curr_c
    
        mover_colors  = {c for c, m in movement.items() if m > 5}
        target_colors = {c for c, m in movement.items() if m == 0}
        return mover_colors, target_colors

    def _movement_fallback_search(self, level_idx, max_states=500000, prev_solution=None, time_budget=None):
        """v16/v20 movement fallback search grafted behind the v19 solver."""
        if not self.game_cls:
            return None
        budget = self.bfs_timeout if time_budget is None else max(1.0, float(time_budget))
        logger.info(f"BFS L{level_idx}: movement fallback budget={budget:.1f}s")
        self._warmup_prefix = []

        game = self.game_cls()
        try:
            game.set_level(level_idx)
        except Exception:
            pass
        game.perform_action(ActionInput(id=GameAction.RESET), raw=True)

        r0 = game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
        if not r0.frame:
            return None
        f0 = np.array(r0.frame[-1])
        bg = int(np.bincount(f0.flatten(), minlength=16).argmax())

        # v9: Try solution transfer from previous level first
        if prev_solution and level_idx > 0:
            transfer_result = self._try_transfer(game, level_idx, prev_solution, f0)
            if transfer_result:
                return transfer_result

        # Phase 1: Scan for effective actions
        actions = self._scan_actions(game, f0, bg)

        # v14 FIX 1: Warm-up unlock — if no actions found, try a warm-up action then re-scan
        if not actions:
            logger.info(f"BFS L{level_idx}: 0 actions found, trying warm-up unlock")
            avail = game._available_actions
            for warmup_id in [a for a in avail if a <= 4]:  # try directional as warm-up
                g_warmup = copy.deepcopy(game)
                try:
                    g_warmup.perform_action(ActionInput(id=GameAction.from_id(warmup_id)), raw=True)
                    f_after = np.array(g_warmup.get_pixels(0, 0, 64, 64))
                    # Re-scan from warmed-up state
                    warmup_actions = self._scan_actions(g_warmup, f_after, bg)
                    if warmup_actions:
                        logger.info(f"BFS L{level_idx}: UNLOCKED with ACTION{warmup_id}! {len(warmup_actions)} actions found")
                        game = g_warmup  # use warmed-up game as new start
                        f0 = f_after
                        actions = warmup_actions
                        # Prepend warm-up to any solution found
                        self._warmup_prefix = [(warmup_id, None)]
                        break
                except:
                    pass

        logger.info(f"BFS L{level_idx}: {len(actions)} effective actions (after dedup)")
        if not actions:
            return None

        # v16: Probe trigger fields BEFORE main BFS for better state distinction
        trigger_fields = None
        raw_hidden = self._probe_hidden_fields(game, actions)
        if raw_hidden:
            clock_fields = set()
            if actions:
                try:
                    g_t1 = copy.deepcopy(game)
                    ai_t = ActionInput(id=GameAction.from_id(actions[0][0]), data=actions[0][1]) if actions[0][1] else ActionInput(id=GameAction.from_id(actions[0][0]))
                    g_t1.perform_action(ai_t, raw=True)
                    g_t2 = copy.deepcopy(g_t1)
                    g_t2.perform_action(ai_t, raw=True)
                    for fld in raw_hidden:
                        v1 = getattr(g_t1, fld, None)
                        v2 = getattr(g_t2, fld, None)
                        if v1 != v2:
                            clock_fields.add(fld)
                except:
                    pass
            trigger_fields = [fld for fld in raw_hidden if fld not in clock_fields]
            if not trigger_fields:
                trigger_fields = None
            else:
                logger.info(f"BFS L{level_idx}: trigger fields for hash: {trigger_fields}")

        # v12: Detect win field + counter direction for A* priority
        win_field = self._extract_win_field()
        counter_dir = 0  # 0=unknown, +1=maximize, -1=minimize
        win_initial = None
        if win_field:
            win_initial = getattr(game, win_field, None)
            if isinstance(win_initial, (int, float)):
                for act_id, data in actions[:5]:
                    g_probe = copy.deepcopy(game)
                    try:
                        ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                        g_probe.perform_action(ai, raw=True)
                        new_val = getattr(g_probe, win_field, win_initial)
                        if isinstance(new_val, (int, float)) and new_val != win_initial:
                            source = open(self.game_path).read()
                            if f'{win_field} >=' in source or f'{win_field} >' in source:
                                counter_dir = +1
                            elif f'{win_field} <=' in source or f'{win_field} <' in source:
                                counter_dir = -1
                            break
                    except:
                        pass
            if counter_dir != 0:
                logger.info(f"BFS L{level_idx}: counter detected: {win_field}={win_initial}, dir={'max' if counter_dir>0 else 'min'}")
                if trigger_fields and win_field not in trigger_fields:
                    trigger_fields.append(win_field)
                elif not trigger_fields:
                    trigger_fields = [win_field]

        # v16: Plain BFS first (with trigger fields in hash), counter A* as fallback
        use_counter_priority = False
        visited = set()
        h0 = self._state_hash(game, f0, trigger_fields)
        visited.add(h0)
        t0 = time.time()
        explored = 0
        fifo_counter = 0

        if use_counter_priority:
            # v12: Lexicographic A* — (counter_rank, depth, fifo_id)
            initial_counter = getattr(game, win_field, 0)
            if not isinstance(initial_counter, (int, float)):
                initial_counter = 0
            counter_rank = -initial_counter * counter_dir  # lower = better
            heap = [(counter_rank, 0, fifo_counter, copy.deepcopy(game), [])]
            fifo_counter += 1

            while heap and explored < max_states and (time.time() - t0) < budget:
                cr, depth, _, g, hist = heapq.heappop(heap)
                for act_id, data in actions:
                    g2 = copy.deepcopy(g)
                    try:
                        ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                        r = g2.perform_action(ai, raw=True)
                    except: continue
                    explored += 1
                    if not r.frame: continue
                    f = np.array(r.frame[-1])
                    # Include win field in hash for counter games
                    wv = getattr(g2, win_field, '')
                    h = (self._state_hash(g2, f, None), win_field, wv)
                    if h in visited: continue
                    visited.add(h)
                    new_hist = hist + [(act_id, data)]
                    if r.levels_completed > level_idx or g2._current_level_index > level_idx:
                        logger.info(f"BFS L{level_idx}: SOLVED (A*) in {len(new_hist)} actions ({explored} explored, {time.time()-t0:.1f}s)")
                        self.solutions[level_idx] = new_hist
                        return new_hist
                    cv = getattr(g2, win_field, 0)
                    new_cr = -(cv if isinstance(cv, (int,float)) else 0) * counter_dir
                    fifo_counter += 1
                    if depth < 30:
                        heapq.heappush(heap, (new_cr, depth+1, fifo_counter, g2, new_hist))
        else:
            # Standard BFS with trigger-aware hashing
            queue = deque()
            queue.append((copy.deepcopy(game), [], 0))
            while queue and explored < max_states and (time.time() - t0) < budget:
                g, hist, depth = queue.popleft()
                for act_id, data in actions:
                    g2 = copy.deepcopy(g)
                    try:
                        ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                        r = g2.perform_action(ai, raw=True)
                    except: continue
                    explored += 1
                    if not r.frame: continue
                    f = np.array(r.frame[-1])
                    h = self._state_hash(g2, f, trigger_fields)
                    if h in visited: continue
                    visited.add(h)
                    new_hist = hist + [(act_id, data)]
                    if r.levels_completed > level_idx or g2._current_level_index > level_idx:
                        logger.info(f"BFS L{level_idx}: SOLVED in {len(new_hist)} actions ({explored} explored, {time.time()-t0:.1f}s)")
                        sol = self._warmup_prefix + new_hist
                        self.solutions[level_idx] = sol
                        return sol
                    if depth < 30:
                        queue.append((g2, new_hist, depth + 1))

        elapsed_first = time.time() - t0
        logger.info(f"BFS L{level_idx}: first pass timeout ({explored} explored, {len(visited)} unique, {elapsed_first:.1f}s)")

        # v16: Counter A* fallback — only runs AFTER plain BFS fails, only when counter detected
        if counter_dir != 0 and win_field and elapsed_first < budget * 0.6:
            remaining_ca = max(5, budget - elapsed_first)
            logger.info(f"BFS L{level_idx}: trying counter A* fallback ({win_field}, dir={'max' if counter_dir>0 else 'min'}, {remaining_ca:.0f}s)")
            game_ca = self.game_cls()
            try:
                game_ca.set_level(level_idx)
            except Exception:
                pass
            game_ca.perform_action(ActionInput(id=GameAction.RESET), raw=True)
            game_ca.perform_action(ActionInput(id=GameAction.RESET), raw=True)
            f0_ca = np.array(game_ca.get_pixels(0, 0, 64, 64))
            initial_counter = getattr(game_ca, win_field, 0)
            if not isinstance(initial_counter, (int, float)):
                initial_counter = 0
            visited_ca = set()
            h0_ca = self._state_hash(game_ca, f0_ca, trigger_fields)
            visited_ca.add(h0_ca)
            counter_rank = -initial_counter * counter_dir
            fifo_ca = 0
            heap_ca = [(counter_rank, 0, fifo_ca, copy.deepcopy(game_ca), [])]
            fifo_ca += 1
            t0_ca = time.time()
            explored_ca = 0
            while heap_ca and explored_ca < max_states and (time.time() - t0_ca) < remaining_ca:
                cr, depth, _, g, hist = heapq.heappop(heap_ca)
                for act_id, data in actions:
                    g2 = copy.deepcopy(g)
                    try:
                        ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                        r = g2.perform_action(ai, raw=True)
                    except: continue
                    explored_ca += 1
                    if not r.frame: continue
                    f = np.array(r.frame[-1])
                    h = self._state_hash(g2, f, trigger_fields)
                    if h in visited_ca: continue
                    visited_ca.add(h)
                    new_hist = hist + [(act_id, data)]
                    if r.levels_completed > level_idx or g2._current_level_index > level_idx:
                        logger.info(f"BFS L{level_idx}: SOLVED (counter A*) in {len(new_hist)} actions ({explored_ca} explored, {time.time()-t0_ca:.1f}s)")
                        sol = self._warmup_prefix + new_hist
                        self.solutions[level_idx] = sol
                        return sol
                    cv = getattr(g2, win_field, 0)
                    new_cr = -(cv if isinstance(cv, (int, float)) else 0) * counter_dir
                    fifo_ca += 1
                    if depth < 40:
                        heapq.heappush(heap_ca, (new_cr, depth + 1, fifo_ca, g2, new_hist))
            logger.info(f"BFS L{level_idx}: counter A* done ({explored_ca} explored, {len(visited_ca)} unique, {time.time()-t0_ca:.1f}s)")

        # v13: ACMD Trigger Finder — when pixels alias, use internal state delta as priority
        # (CHRONOS Gemini T34, n=0.109: "Action-Conditional Masked RAM Delta Priority")
        if len(visited) < 100 and elapsed_first < budget * 0.8:
            hidden_fields = self._probe_hidden_fields(game, actions)
            if hidden_fields:
                logger.info(f"BFS L{level_idx}: ACMD trigger search with fields: {hidden_fields}")

                # Pre-compute clock mask: fields that change on NO-OP (timers, not triggers)
                clock_fields = set()
                g_noop = copy.deepcopy(game)
                snap_before = {f: getattr(g_noop, f, None) for f in hidden_fields}
                try:
                    # Try a no-op: perform same action twice, see what auto-changes
                    if actions:
                        g_noop2 = copy.deepcopy(g_noop)
                        ai = ActionInput(id=GameAction.from_id(actions[0][0]), data=actions[0][1]) if actions[0][1] else ActionInput(id=GameAction.from_id(actions[0][0]))
                        g_noop2.perform_action(ai, raw=True)
                        g_noop3 = copy.deepcopy(g_noop2)
                        g_noop3.perform_action(ai, raw=True)
                        for f in hidden_fields:
                            v1 = getattr(g_noop2, f, None)
                            v2 = getattr(g_noop3, f, None)
                            if v1 == v2:  # didn't change between identical actions → not a clock
                                pass
                            else:
                                clock_fields.add(f)
                except: pass
                trigger_fields = [f for f in hidden_fields if f not in clock_fields]
                if not trigger_fields:
                    trigger_fields = hidden_fields  # fallback: use all

                # ACMD priority search: promote actions that change trigger fields
                game2 = self.game_cls()
                try:
                    game2.set_level(level_idx)
                except Exception:
                    pass
                game2.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                r0_2 = game2.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                if not r0_2.frame:
                    return None
                f0_2 = np.array(r0_2.frame[-1])

                visited2 = set()
                init_state = {f: getattr(game2, f, None) for f in trigger_fields}
                h0_2 = self._state_hash(game2, f0_2, trigger_fields)
                visited2.add(h0_2)
                fifo2 = 0
                # Priority: (negative_trigger_delta, depth, fifo) — lower = better
                heap2 = [(0, 0, fifo2, copy.deepcopy(game2), [])]
                fifo2 += 1

                t0_2 = time.time()
                explored2 = 0
                remaining = max(5, budget - elapsed_first)

                while heap2 and explored2 < max_states and (time.time() - t0_2) < remaining:
                    neg_delta, depth, _, g, hist = heapq.heappop(heap2)

                    for act_id, data in actions:
                        g2 = copy.deepcopy(g)
                        try:
                            ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                            r = g2.perform_action(ai, raw=True)
                        except: continue
                        explored2 += 1
                        if not r.frame: continue
                        f = np.array(r.frame[-1])
                        h = self._state_hash(g2, f, trigger_fields)
                        if h in visited2: continue
                        visited2.add(h)
                        new_hist = hist + [(act_id, data)]

                        if r.levels_completed > level_idx or g2._current_level_index > level_idx:
                            logger.info(f"BFS L{level_idx}: SOLVED (ACMD) in {len(new_hist)} actions ({explored2} explored, {time.time()-t0_2:.1f}s)")
                            self.solutions[level_idx] = new_hist
                            return new_hist

                        # Compute trigger delta: how much did trigger fields change?
                        pixels_changed = np.sum(f0_2 != f) > 0
                        trigger_delta = 0
                        for tf in trigger_fields:
                            cv = getattr(g2, tf, None)
                            iv = init_state.get(tf)
                            if isinstance(cv, (int, float)) and isinstance(iv, (int, float)):
                                trigger_delta += abs(cv - iv)
                            elif cv != iv:
                                trigger_delta += 1

                        # ACMD priority: PROMOTE if trigger changed, PRUNE if nothing changed
                        if not pixels_changed and trigger_delta == 0:
                            continue  # true no-op: prune completely
                        # Lower priority = explored first. Negative delta = more trigger progress
                        priority = -trigger_delta
                        fifo2 += 1
                        if depth < 40:
                            heapq.heappush(heap2, (priority, depth + 1, fifo2, g2, new_hist))

                logger.info(f"BFS L{level_idx}: ACMD finished ({explored2} explored, {len(visited2)} unique, {time.time()-t0_2:.1f}s)")

        # v16: Sprite permutation for pure-click games with few targets
        elapsed_perm_start = time.time() - t0
        click_actions = [a for a in actions if a[0] == 6]
        non_click = [a for a in actions if a[0] != 6]
        if not non_click and 1 <= len(click_actions) <= 8 and (budget - elapsed_perm_start) > 10:
            n_perms = 1
            for i in range(1, len(click_actions)+1): n_perms *= i
            logger.info(f"BFS L{level_idx}: trying sprite permutation ({len(click_actions)} clicks, {n_perms} perms)")
            t0_perm = time.time()
            perm_timeout = min(60, budget - elapsed_perm_start)
            for perm in permutations(range(len(click_actions))):
                if time.time() - t0_perm > perm_timeout:
                    break
                g_perm = copy.deepcopy(game)
                hist_perm = []
                solved = False
                for idx in perm:
                    act_id, data = click_actions[idx]
                    try:
                        ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                        r = g_perm.perform_action(ai, raw=True)
                        hist_perm.append((act_id, data))
                        if r.levels_completed > level_idx or g_perm._current_level_index > level_idx:
                            logger.info(f"BFS L{level_idx}: SOLVED (permutation) in {len(hist_perm)} actions")
                            sol = self._warmup_prefix + hist_perm
                            self.solutions[level_idx] = sol
                            return sol
                    except:
                        break
            logger.info(f"BFS L{level_idx}: permutation exhausted ({time.time()-t0_perm:.1f}s)")

        # v14 FIX 2: IDDFS for deep directional games (low branching, deep solution)
        elapsed_total = time.time() - t0
        remaining_time = max(5, budget - elapsed_total)
        if len(actions) <= 6 and remaining_time > 30:
            logger.info(f"BFS L{level_idx}: trying IDDFS (branching={len(actions)}, {remaining_time:.0f}s remaining)")
            game3 = self.game_cls()
            try:
                game3.set_level(level_idx)
            except Exception:
                pass
            game3.perform_action(ActionInput(id=GameAction.RESET), raw=True)
            game3.perform_action(ActionInput(id=GameAction.RESET), raw=True)
            t0_3 = time.time()
            for max_depth in range(10, 60):
                if time.time() - t0_3 > remaining_time:
                    break
                # DFS with depth limit + path-based cycle detection
                stack = [(copy.deepcopy(game3), [], set())]
                explored3 = 0
                while stack and (time.time() - t0_3) < remaining_time:
                    g, hist, path_hashes = stack.pop()
                    if len(hist) >= max_depth:
                        continue
                    for act_id, data in actions:
                        g2 = copy.deepcopy(g)
                        try:
                            ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                            r = g2.perform_action(ai, raw=True)
                        except: continue
                        explored3 += 1
                        if not r.frame: continue
                        f = np.array(r.frame[-1])
                        h = self._state_hash(g2, f, trigger_fields)
                        if h in path_hashes: continue
                        new_hist = hist + [(act_id, data)]
                        if r.levels_completed > level_idx or g2._current_level_index > level_idx:
                            logger.info(f"BFS L{level_idx}: SOLVED (IDDFS depth={max_depth}) in {len(new_hist)} actions ({explored3} explored, {time.time()-t0_3:.1f}s)")
                            sol = self._warmup_prefix + new_hist
                            self.solutions[level_idx] = sol
                            return sol
                        new_path = path_hashes | {h}
                        stack.append((g2, new_hist, new_path))
            logger.info(f"BFS L{level_idx}: IDDFS exhausted (depth={max_depth}, {time.time()-t0_3:.1f}s)")

        # v17: Beam search fallback — guided by trigger + pixel progress
        elapsed_bs = time.time() - t0
        remaining_bs = max(5, budget - elapsed_bs)
        if 2 <= len(actions) <= 15 and remaining_bs > 20:
            logger.info(f"BFS L{level_idx}: trying beam search (b={len(actions)}, {remaining_bs:.0f}s)")
            bw = min(200, max(20, max_states // (len(actions) * 50)))
            game_b = self.game_cls()
            try:
                game_b.set_level(level_idx)
            except Exception:
                pass
            game_b.perform_action(ActionInput(id=GameAction.RESET), raw=True)
            game_b.perform_action(ActionInput(id=GameAction.RESET), raw=True)
            f0_b = np.array(game_b.get_pixels(0, 0, 64, 64))
            beam = [(copy.deepcopy(game_b), [])]
            t0_b = time.time()
            vis_b = set()
            vis_b.add(self._state_hash(game_b, f0_b, trigger_fields))
            for bd in range(60):
                if time.time() - t0_b > remaining_bs or not beam:
                    break
                cands = []
                for g_b, hist_b in beam:
                    for act_id, data in actions:
                        g2 = copy.deepcopy(g_b)
                        try:
                            ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                            r = g2.perform_action(ai, raw=True)
                        except Exception:
                            continue
                        if not r.frame:
                            continue
                        f = np.array(r.frame[-1])
                        h = self._state_hash(g2, f, trigger_fields)
                        if h in vis_b:
                            continue
                        vis_b.add(h)
                        nh = hist_b + [(act_id, data)]
                        if r.levels_completed > level_idx or g2._current_level_index > level_idx:
                            logger.info(f"BFS L{level_idx}: SOLVED (beam d={bd}) in {len(nh)} acts")
                            sol = self._warmup_prefix + nh
                            self.solutions[level_idx] = sol
                            return sol
                        pdiff = float(np.sum(f != f0_b)) / 4096.0
                        tscore = 0.0
                        if trigger_fields:
                            for tf in trigger_fields:
                                cv = getattr(g2, tf, None)
                                iv = getattr(game_b, tf, None)
                                if isinstance(cv, (int, float)) and isinstance(iv, (int, float)):
                                    tscore += abs(cv - iv)
                        cands.append((tscore * 10.0 + pdiff, g2, nh))
                if not cands:
                    break
                cands.sort(key=lambda x: x[0], reverse=True)
                beam = [(g_b, h_b) for _, g_b, h_b in cands[:bw]]
            logger.info(f"BFS L{level_idx}: beam done ({len(vis_b)} unique, {time.time()-t0_b:.1f}s)")

        return None
    
    def solve_level(self, level_idx, max_states=500000, prev_solution=None, goal_heuristic=None):
        """Find optimal solution for a level via BFS (Memory Optimised via Action Replay)."""
        if not self.game_cls:
            return None
        method_t0 = time.time()

        game = self.game_cls()
        game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
        r0 = game.perform_action(ActionInput(id=GameAction.RESET), raw=True)

        # Advance to target level by replaying previous solutions
        last_r = r0
        for prev_idx in range(level_idx):
            prev_sol = self.solutions.get(prev_idx)
            if not prev_sol:
                return None
            for act_id, data in prev_sol:
                ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                last_r = game.perform_action(ai, raw=True)

        if not last_r.frame:
            return None
        f0 = np.array(last_r.frame[-1])
        bg = int(np.bincount(f0.flatten(), minlength=16).argmax())

        # Try solution transfer from previous level first
        if prev_solution and level_idx > 0:
            transfer_result = self._try_transfer(game, level_idx, prev_solution, f0)
            if transfer_result:
                return transfer_result

        # Phase 1: Scan for effective actions
        actions = self._scan_actions(game, f0, bg)

        # Warm-up unlock for locked initial states (sc25-type)
        if not actions:
            avail = game._available_actions
            # Try all non-reset actions as warmup, including clicks
            warmup_candidates = [a for a in avail if 1 <= a <= 5]
            # Also try click actions from _get_valid_actions if available
            if 6 in avail and hasattr(game, '_get_valid_actions'):
                try:
                    for va in game._get_valid_actions():
                        act_id = va.id._value_ if hasattr(va.id, '_value_') else int(va.id)
                        if act_id == 6:
                            g_warmup = _fast_deepcopy(game)
                            try:
                                g_warmup.perform_action(va, raw=True)
                                f_after = np.array(g_warmup.perform_action(
                                    ActionInput(id=GameAction.ACTION1), raw=True).frame[-1])
                                warmup_actions = self._scan_actions(g_warmup, f_after, bg)
                                if warmup_actions:
                                    logger.info(f"BFS L{level_idx}: UNLOCKED with click! {len(warmup_actions)} actions")
                                    game = g_warmup; f0 = f_after; actions = warmup_actions
                                    break
                            except:
                                pass
                except:
                    pass
            if not actions:
                for warmup_id in [a for a in avail if a <= 4]:
                    g_warmup = _fast_deepcopy(game)
                    try:
                        g_warmup.perform_action(ActionInput(id=GameAction.from_id(warmup_id)), raw=True)
                        f_after = np.array(g_warmup.get_pixels(0, 0, 64, 64))
                        warmup_actions = self._scan_actions(g_warmup, f_after, bg)
                        if warmup_actions:
                            logger.info(f"BFS L{level_idx}: UNLOCKED with ACTION{warmup_id}! {len(warmup_actions)} actions")
                            game = g_warmup; f0 = f_after; actions = warmup_actions
                            break
                    except:
                        pass

        logger.info(f"BFS L{level_idx}: {len(actions)} effective actions")
        if not actions:
            return None

       # ==========================================
        # Phase 2: A* with goal heuristic from prev level
        # ==========================================
        import heapq
        hidden_fields = None
        transient_fields = self._detect_transient_fields(game, actions)
        visited = set()
        h0 = self._state_hash(game, f0, None, transient_fields=transient_fields)
        visited.add(h0)
        base_game = _fast_deepcopy(game)

        hfn = goal_heuristic if goal_heuristic is not None else (lambda f, game=None: 0)
        # If heuristic is flat (no goal_heuristic provided or indicator-based),
        # probe mover/target colors and use distance heuristic instead
        
        _hfn_uses_game = goal_heuristic is not None
        counter = 0
        pq = [(hfn(f0, game) * 10, 0, counter, [], base_game)]
        t0 = time.time()
        explored = 0

        while pq and explored < max_states and (time.time() - t0) < self.bfs_timeout:
            f_score, g_score, _, hist, node_game = heapq.heappop(pq)
            
            for act_id, data in actions:
                g2 = _fast_deepcopy(node_game)
                try:
                    ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                    r = g2.perform_action(ai, raw=True)
                except:
                    continue
                explored += 1

                if not r.frame:
                    continue
                f = np.array(r.frame[-1])
                h = self._state_hash(g2, f, hidden_fields, transient_fields=transient_fields)
                if h in visited:
                    continue
                visited.add(h)

                new_hist = hist + [(act_id, data)]
                new_g = g_score + 1

                if r.levels_completed > level_idx or g2._current_level_index > level_idx:
                    elapsed = time.time() - t0
                    logger.info(f"BFS L{level_idx}: SOLVED (A*) in {len(new_hist)} actions ({explored} explored, {elapsed:.1f}s)")
                    self.solutions[level_idx] = new_hist
                    return new_hist

                h_val = hfn(f, g2 if _hfn_uses_game else None) * 10 
                counter += 1
                heapq.heappush(pq, (new_g + h_val, new_g, counter, new_hist, g2))

        elapsed_first = time.time() - t0
        logger.info(f"BFS L{level_idx}: first pass timeout ({explored} explored, {len(visited)} unique, {elapsed_first:.1f}s)")
        self.timed_out_levels.add(level_idx)
        # Dynamic action rescan BFS — triggers when state space exhausted quickly
        # indicating actions expand as state evolves (e.g. flood fill games)
        exhausted_quickly = len(pq) == 0 and elapsed_first < self.bfs_timeout * 0.5
        if exhausted_quickly:
            logger.info(f"BFS L{level_idx}: queue exhausted early — retrying with dynamic action rescan")
            visited_d = set()
            visited_d.add(self._state_hash(base_game, f0, hidden_fields, transient_fields=transient_fields))
            queue_d = deque()
            queue_d.append(([], 0, base_game))
            t0_d = time.time()
            explored_d = 0
            remaining_d = max(30, self.bfs_timeout - elapsed_first)
            current_actions = list(actions)

            while queue_d and explored_d < max_states * 10 and (time.time() - t0_d) < remaining_d:
                hist_d, depth_d, node_game_d = queue_d.popleft()

                for act_id, data in current_actions:
                    g2_d = _fast_deepcopy(node_game_d)
                    try:
                        ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                        r = g2_d.perform_action(ai, raw=True)
                    except:
                        continue
                    explored_d += 1
                    if not r.frame:
                        continue
                    f2_d = np.array(r.frame[-1])
                    h_d = self._state_hash(g2_d, f2_d, hidden_fields, transient_fields=transient_fields)
                    if h_d in visited_d:
                        continue
                    visited_d.add(h_d)
                    # Rescan from child state to find newly unlocked actions
                    try:
                        new_acts = self._scan_actions(g2_d, f2_d, bg)
                        added = [a for a in new_acts if a not in current_actions]
                        if added:
                            logger.info(f"BFS L{level_idx}: rescan found {len(added)} new actions at depth {depth_d}")
                            current_actions.extend(added)
                    except:
                        pass
                    new_hist_d = hist_d + [(act_id, data)]
                    if r.levels_completed > level_idx or g2_d._current_level_index > level_idx:
                        logger.info(f"BFS L{level_idx}: SOLVED (dynamic rescan) in {len(new_hist_d)} actions ({explored_d} explored)")
                        self.solutions[level_idx] = new_hist_d
                        return new_hist_d
                    if depth_d < 30:
                        queue_d.append((new_hist_d, depth_d + 1, g2_d))

            logger.info(f"BFS L{level_idx}: dynamic rescan also failed ({explored_d} explored)")

        # Smart early exit — game may be too expensive to BFS
        if explored < 20 and elapsed_first > 10.0:
            logger.info(f"BFS L{level_idx}: early exit (only {explored} explored in {elapsed_first:.1f}s) — handing off to CNN")
            return None

        # If too few unique states found → hidden state detected → retry with probed fields
        if explored > 0 and (len(visited) < 200 or explored / len(visited) > 5) and elapsed_first < self.bfs_timeout * 0.8:
            hidden_fields = self._probe_hidden_fields(game, actions)
            if hidden_fields:
                logger.info(f"BFS L{level_idx}: RETRY with hidden fields: {hidden_fields}")

                # FIX 3: Use exactly 2 RESET calls (not 3) to match the first pass baseline
                game2 = self.game_cls()
                game2.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                last_r2 = game2.perform_action(ActionInput(id=GameAction.RESET), raw=True)

                for prev_idx in range(level_idx):
                    prev_sol = self.solutions.get(prev_idx)
                    if not prev_sol:
                        return None
                    for act_id, data in prev_sol:
                        ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                        last_r2 = game2.perform_action(ai, raw=True)

                if not last_r2.frame:
                    return None
                f0_2 = np.array(last_r2.frame[-1])
                h0_2 = self._state_hash(game2, f0_2, hidden_fields, transient_fields=transient_fields)

                base_game2 = _fast_deepcopy(game2)
                visited2 = set()
                visited2.add(h0_2)
                queue2 = deque()
                queue2.append(([], 0, base_game2))

                t0_2 = time.time()
                explored2 = 0
                remaining = max(30, self.bfs_timeout - elapsed_first)

                while queue2 and explored2 < max_states and (time.time() - t0_2) < remaining:
                    hist, depth, node_game2 = queue2.popleft()

                    for act_id, data in actions:
                        g2 = _fast_deepcopy(node_game2)
                        try:
                            ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                            r = g2.perform_action(ai, raw=True)
                        except:
                            continue
                        explored2 += 1

                        if not r.frame:
                            continue
                        f = np.array(r.frame[-1])
                        h = self._state_hash(g2, f, hidden_fields, transient_fields=transient_fields)
                        if h in visited2:
                            continue
                        visited2.add(h)

                        new_hist = hist + [(act_id, data)]

                        if r.levels_completed > level_idx or g2._current_level_index > level_idx:
                            logger.info(f"BFS L{level_idx}: SOLVED (hidden retry) in {len(new_hist)} actions ({explored2} explored)")
                            self.solutions[level_idx] = new_hist
                            return new_hist

                        if depth < 30:
                            queue2.append((new_hist, depth + 1, g2))

                logger.info(f"BFS L{level_idx}: hidden retry also failed ({explored2} explored, {len(visited2)} unique)")

        remaining_fb = self.bfs_timeout - (time.time() - method_t0)
        if remaining_fb > 8:
            try:
                fb = self._movement_fallback_search(
                    level_idx,
                    max_states=max(1000, max_states // 2),
                    prev_solution=prev_solution,
                    time_budget=remaining_fb,
                )
                if fb:
                    return fb
            except Exception as e:
                logger.warning(f"BFS L{level_idx}: movement fallback failed: {e}")

        return None

    def _try_transfer(self, game, level_idx, prev_solution, f1):
        """v13: Affine transfer with scale detection + action count multiplier."""
        try:
            # Try executing prev solution directly (sometimes levels share exact solution)
            g = copy.deepcopy(game)
            for i, (act_id, data) in enumerate(prev_solution):
                try:
                    ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                    r = g.perform_action(ai, raw=True)
                    if r.levels_completed > level_idx or g._current_level_index > level_idx:
                        logger.info(f"BFS L{level_idx}: TRANSFER SUCCESS (direct replay, {i+1} actions)")
                        sol = prev_solution[:i+1]
                        self.solutions[level_idx] = sol
                        return sol
                except:
                    break

            # Try object-relative transfer (CHRONOS Opus T11)
            prev_game = self.game_cls()
            prev_game.set_level(level_idx - 1)
            prev_game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
            r_prev = prev_game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
            if not r_prev.frame:
                return None
            f0 = np.array(r_prev.frame[-1])
            bg = int(np.bincount(f0.flatten(), minlength=16).argmax())

            # Extract objects from both levels
            def get_objects(frame, bg_c):
                objs = []
                for c in range(16):
                    if c == bg_c:
                        continue
                    mask = (frame == c)
                    npix = int(np.sum(mask))
                    if npix < 2:
                        continue
                    ys, xs = np.where(mask)
                    objs.append({'color': c, 'cx': float(np.mean(xs)), 'cy': float(np.mean(ys)), 'n': npix})
                return sorted(objs, key=lambda o: (o['color'], -o['n']))

            objs_prev = get_objects(f0, bg)
            objs_curr = get_objects(f1, bg)

            if not objs_prev or not objs_curr:
                return None

            # Match objects by color + relative size
            matched = []
            for op in objs_prev:
                best = None
                best_dist = float('inf')
                for oc in objs_curr:
                    if oc['color'] == op['color'] and abs(oc['n'] - op['n']) < max(op['n'], oc['n']) * 0.5:
                        d = abs(oc['cx'] - op['cx']) + abs(oc['cy'] - op['cy'])
                        if d < best_dist:
                            best_dist = d
                            best = oc
                if best:
                    matched.append((op, best))

            if not matched:
                return None

            # Compute offset
            dx = np.mean([m[1]['cx'] - m[0]['cx'] for m in matched])
            dy = np.mean([m[1]['cy'] - m[0]['cy'] for m in matched])

            # Apply offset to click actions
            transferred = []
            for act_id, data in prev_solution:
                if data and 'x' in data:
                    new_data = dict(data)
                    new_data['x'] = max(0, min(63, int(data['x'] + dx)))
                    new_data['y'] = max(0, min(63, int(data['y'] + dy)))
                    transferred.append((act_id, new_data))
                else:
                    transferred.append((act_id, data))

            # Validate transferred solution
            g = copy.deepcopy(game)
            for i, (act_id, data) in enumerate(transferred):
                try:
                    ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                    r = g.perform_action(ai, raw=True)
                    if r.levels_completed > level_idx or g._current_level_index > level_idx:
                        logger.info(f"BFS L{level_idx}: TRANSFER SUCCESS (offset dx={dx:.0f},dy={dy:.0f}, {i+1} actions)")
                        sol = transferred[:i+1]
                        self.solutions[level_idx] = sol
                        return sol
                except:
                    break

            # v13: If offset transfer failed, try action-count multiplier (CHRONOS T28)
            # L1 might need same actions repeated more times
            for multiplier in [2, 3, 4]:
                expanded = []
                for act_id, data in prev_solution:
                    for _ in range(int(multiplier)):
                        if data:
                            new_data = dict(data)
                            new_data['x'] = max(0, min(63, int(data.get('x', 32) + dx)))
                            new_data['y'] = max(0, min(63, int(data.get('y', 32) + dy)))
                            expanded.append((act_id, new_data))
                        else:
                            expanded.append((act_id, data))
                g = copy.deepcopy(game)
                for i, (act_id, data) in enumerate(expanded):
                    try:
                        ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                        r = g.perform_action(ai, raw=True)
                        if r.levels_completed > level_idx or g._current_level_index > level_idx:
                            logger.info(f"BFS L{level_idx}: TRANSFER SUCCESS (multiplier={multiplier}, {i+1} actions)")
                            sol = expanded[:i+1]
                            self.solutions[level_idx] = sol
                            return sol
                    except:
                        break

        except Exception as e:
            logger.warning(f"BFS transfer failed: {e}")
        return None


def find_game_source_and_class(game_id, arc_env=None):
    """Find the game .py file and class name."""
    import re

    # game_id format: sk48-d8078629
    # file lives at: .../environment_files/sk48/d8078629/sk48.py
    parts = game_id.split('-', 1)
    gid = parts[0]                          # e.g. sk48
    guid_suffix = parts[1] if len(parts) > 1 else ''  # e.g. d8078629

    # Primary: competition path on Kaggle
    competition_path = (
        f"/kaggle/input/competitions/arc-prize-2026-arc-agi-3"
        f"/environment_files/{gid}/{guid_suffix}/{gid}.py"
    )
    if os.path.exists(competition_path):
        src = competition_path
        content = open(src).read()[:2000]
        m = re.search(r'class\s+(\w+)\s*\(', content)
        cls_name = m.group(1) if m else gid[0].upper() + gid[1:]
        logger.info(f"BFS: found game source at {src}, class={cls_name}")
        return src, cls_name

    # Fallback: broad glob search
    for pattern in [
        f"/kaggle/input/**/{gid}.py",
        f"/tmp/**/{gid}.py",
        f"/kaggle/working/**/{gid}.py",
    ]:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            src = matches[0]
            content = open(src).read()[:2000]
            m = re.search(r'class\s+(\w+)\s*\(', content)
            cls_name = m.group(1) if m else gid[0].upper() + gid[1:]
            logger.info(f"BFS: found game source at {src}, class={cls_name}")
            return src, cls_name

    logger.warning(f"BFS: game source not found for {game_id}")
    return None, gid[0].upper() + gid[1:]


# ==================== CNN FALLBACK ====================

class CBAM(nn.Module):
    def __init__(s, ch, r=16):
        super().__init__()
        s.fc1=nn.Linear(ch,max(ch//r,4)); s.fc2=nn.Linear(max(ch//r,4),ch)
        s.sp=nn.Conv2d(2,1,7,padding=3)
    def forward(s, x):
        B,C,H,W=x.shape
        w=torch.sigmoid(s.fc2(F.relu(s.fc1(x.mean(dim=[2,3]))))); x=x*w.view(B,C,1,1)
        a=torch.sigmoid(s.sp(torch.cat([x.max(1,keepdim=True)[0],x.mean(1,keepdim=True)],1)))
        return x*a

class ActionEffectAttention(nn.Module):
    def __init__(s, feat_dim=64, mem_dim=32, n_actions=5):
        super().__init__()
        s.mem_dim=mem_dim
        s.diff_enc=nn.Sequential(nn.Conv2d(1,8,8,stride=8),nn.ReLU(),nn.Conv2d(8,16,4,stride=4),nn.ReLU(),nn.Flatten(),nn.Linear(16*2*2,mem_dim))
        s.q_proj=nn.Linear(feat_dim,mem_dim)
        s.v_proj=nn.Linear(mem_dim+1+n_actions,n_actions)
        s.scale=mem_dim**0.5
    def forward(s, cnn_feat, mem_diffs, mem_actions, mem_rewards):
        B,M=mem_actions.shape
        if M==0:return torch.zeros(B,5,device=cnn_feat.device)
        keys=s.diff_enc(mem_diffs.reshape(B*M,1,64,64)).reshape(B,M,s.mem_dim)
        q=s.q_proj(cnn_feat).unsqueeze(1)
        attn=F.softmax(torch.bmm(q,keys.transpose(1,2))/s.scale,dim=-1)
        act_oh=F.one_hot(mem_actions.clamp(0,4),5).float()
        vals=torch.cat([keys,mem_rewards.unsqueeze(-1),act_oh],dim=-1)
        ctx=torch.bmm(attn,vals).squeeze(1)
        return s.v_proj(ctx)

class ForgeNet(nn.Module):
    def __init__(s, in_ch=26, g=64):
        super().__init__()
        s.g=g
        s.c1=nn.Conv2d(in_ch,32,3,padding=1);s.c2=nn.Conv2d(32,64,3,padding=1)
        s.c3=nn.Conv2d(64,128,3,padding=1);s.c4=nn.Conv2d(128,256,3,padding=1)
        s.attn=CBAM(256);s.ar=nn.Conv2d(256,64,1);s.ap=nn.MaxPool2d(4,4)
        s.af=nn.Linear(64*16*16,256);s.ah=nn.Linear(256,5);s.dr=nn.Dropout(0.15)
        s.cc1=nn.Conv2d(256,128,3,padding=1);s.cc2=nn.Conv2d(128,64,3,padding=1)
        s.cc3=nn.Conv2d(64,32,1);s.cc4=nn.Conv2d(32,1,1)
        s.gp=nn.AdaptiveAvgPool2d(1);s.gf=nn.Linear(256,64)
        s.aea=ActionEffectAttention(feat_dim=64,mem_dim=32,n_actions=5)
    def forward(s, x, mem_diffs=None, mem_actions=None, mem_rewards=None):
        x=F.relu(s.c1(x));x=F.relu(s.c2(x));x=F.relu(s.c3(x));f=F.relu(s.c4(x))
        f=s.attn(f);af=F.relu(s.ar(f));af=s.ap(af).reshape(f.size(0),-1)
        al=s.ah(s.dr(F.relu(s.af(af))))
        cf=F.relu(s.cc1(f));cf=F.relu(s.cc2(cf));cf=F.relu(s.cc3(cf))
        cl=s.cc4(cf).reshape(f.size(0),-1)
        if mem_diffs is not None and mem_actions is not None:
            gf=s.gf(s.gp(f).reshape(f.size(0),-1))
            al=al+s.aea(gf,mem_diffs,mem_actions,mem_rewards)
        return torch.cat([al,cl],1)


def fast_objects(frame, bg, exclude_colours=None, static_mask=None):
    if exclude_colours is None:
        exclude_colours = set()
    objs = []
    for c in range(16):
        if c == bg or c in exclude_colours:
            continue
        if static_mask is not None:
            mask = (frame == c) & ~static_mask
        else:
            mask = (frame == c)
        npix = int(np.sum(mask))
        if npix < 4 or npix > 3000:
            continue
        ys, xs = np.where(mask)
        objs.append((c, float(np.mean(xs)), float(np.mean(ys)), npix,
                     int(xs.max()-xs.min()), int(ys.max()-ys.min()),
                     int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())))
    return objs


def find_composite_objects(objs, proximity=6):
    if not objs:
        return []
    n = len(objs)
    adjacent = [set() for _ in range(n)]
    for i in range(n):
        for j in range(i+1, n):
            oi, oj = objs[i], objs[j]
            x_gap = max(0, max(oi[6], oj[6]) - min(oi[8], oj[8]))
            y_gap = max(0, max(oi[7], oj[7]) - min(oi[9], oj[9]))
            if x_gap <= proximity and y_gap <= proximity:
                adjacent[i].add(j)
                adjacent[j].add(i)
    visited = [False] * n
    groups = []
    for i in range(n):
        if visited[i]:
            continue
        group = []
        stack = [i]
        while stack:
            node = stack.pop()
            if visited[node]:
                continue
            visited[node] = True
            group.append(node)
            stack.extend(adjacent[node] - set(g for g in group))
        groups.append([objs[k] for k in group])
    filtered = []
    for group in groups:
        x_min = min(o[6] for o in group)
        y_min = min(o[7] for o in group)
        x_max = max(o[8] for o in group)
        y_max = max(o[9] for o in group)
        area = (x_max - x_min + 1) * (y_max - y_min + 1)
        if area < 64 * 64 * 0.4:
            filtered.append(group)
    return filtered



# =====================================================================
# FORGE strategy-prior training support
# Consumes an offline JSON prior generated from gaming_strategy_dataset.tar.gz.
# This never executes raw console controls; it only biases ARC ACTION1..ACTION7.
# =====================================================================
def _load_forge_strategy_prior():
    if os.environ.get("FORGE_USE_STRATEGY_PRIOR", "1").strip() in ("0", "false", "False", "no", "NO"):
        return {}
    path = os.environ.get("FORGE_STRATEGY_PRIOR_PATH", "").strip()
    candidates = [path] if path else []
    candidates += [
        "strategy_prior.json",
        "agent_training/strategy_prior.json",
        "/mnt/data/agent_training/strategy_prior.json",
        "/kaggle/working/agent_training/strategy_prior.json",
    ]
    for p in candidates:
        if not p:
            continue
        try:
            if os.path.exists(p):
                with open(p, "r", encoding="utf-8") as f:
                    prior = json.load(f)
                if prior.get("training_policy", {}).get("raw_console_controls_executable") is True:
                    logger.warning("Strategy prior rejected: unsafe raw console control policy")
                    return {}
                logger.info(f"FORGE strategy prior loaded: {p}")
                return prior
        except Exception as e:
            logger.warning(f"FORGE strategy prior load failed for {p}: {e}")
    return {}

def _prior_float(prior, key, default=0.0):
    try:
        return float(prior.get(key, default))
    except Exception:
        return float(default)

# ==================== AGENT ====================

class MyAgent(Agent):
    MAX_ACTIONS = float('inf')
    _MAX_FRAMES = 10

    def __init__(s, *a, **kw):
        super().__init__(*a, **kw)
        seed = int(time.time()*1e6) + hash(s.game_id) % 1000000
        random.seed(seed); np.random.seed(seed%(2**32-1))
        if FORGE_TORCH_AVAILABLE:
            torch.manual_seed(seed%(2**32-1))
        s.start_time = time.time()
        s.device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
        s.G=64; s.IN=26
        s.net=None; s.opt=None
        s.buf=deque(maxlen=50000); s.buf_h=set()
        s.bsz=64; s.tfreq=10
        s.pt=None; s.pai=None; s.pr=None; s.ph=None
        s.cl=-1; s.fhist=deque(maxlen=6); s.la=0
        s.al=[GameAction.ACTION1,GameAction.ACTION2,GameAction.ACTION3,GameAction.ACTION4,GameAction.ACTION5]
        s._wd=False; s._bg=0; s._wm=None
        s._aem_diffs=deque(maxlen=256); s._aem_actions=deque(maxlen=256); s._aem_rewards=deque(maxlen=256)
        s._ckpt_hash=None; s._unproductive=0; s._undo_avail=False
        s._eps=0.15; s._eps_min=0.03; s._eps_decay=0.9997
        s._prev_objs=None; s._obj_moved=0
        # FIX 1: Initialize _visited_hashes so _reward() deduplication works correctly
        s._visited_hashes = set()
        # BFS solver
        s._bfs = None
        s._bfs_solution = None
        s._bfs_step = 0
        s._bfs_tried = False

        # Object model
        s._frame_buffer = []
        s._static_mask = None
        s._dynamic_mask = None
        s._static_ready = False
        s._structural_colours = set()
        s._target_colours = set()
        s._goal_groups = []
        s._bg = 0

        # Offline strategy-prior training from gaming_strategy_dataset.tar.gz.
        # Provides small logit biases and an opening curriculum. It is intentionally
        # weaker than verified BFS and live transition evidence.
        s._strategy_prior = _load_forge_strategy_prior()
        s._prior_strength = float(os.environ.get("FORGE_STRATEGY_PRIOR_STRENGTH", "0.35") or 0.35)
        try:
            vec = s._strategy_prior.get("action_bias_vector_ACTION1_to_ACTION5", [0,0,0,0,0])
            if not isinstance(vec, list) or len(vec) != 5:
                vec = [0,0,0,0,0]
            s._prior_action_bias = [float(x) for x in vec]
            s._prior_click_bias = float(s._strategy_prior.get("click_ACTION6_bias", 0.0))
            s._prior_undo_bias = float(s._strategy_prior.get("undo_ACTION7_bias", 0.0))
            s._prior_opening = [int(x) for x in s._strategy_prior.get("opening_action_ids", []) if 1 <= int(x) <= 5]
        except Exception:
            s._prior_action_bias = [0,0,0,0,0]
            s._prior_click_bias = 0.0
            s._prior_undo_bias = 0.0
            s._prior_opening = []
        if s._strategy_prior:
            logger.info(f"FORGE strategy-prior active: strength={s._prior_strength}, opening={s._prior_opening}, bias={s._prior_action_bias}")

    def append_frame(s, f):
        s.frames.append(f)
        if len(s.frames) > s._MAX_FRAMES: s.frames = s.frames[-s._MAX_FRAMES:]
        if f.guid: s.guid = f.guid
        if hasattr(s, "recorder") and not s.is_playback:
            import json; s.recorder.record(json.loads(f.model_dump_json()))

    def _lvl(s, f): return getattr(f, 'score', None) or f.levels_completed
    def _raw(s, fd): return np.array(fd.frame, dtype=np.int64)[-1]

    def _init_bfs(s):
        """Initialize BFS solver on first call."""
        src, cls = find_game_source_and_class(s.game_id, s.arc_env)
        if src:
            s._bfs = BFSSolver(src, cls, scan_timeout=5, bfs_timeout=180)
            if s._bfs.load():
                logger.info(f"BFS: loaded {cls} from {src}")
            else:
                s._bfs = None
                logger.warning(f"BFS: failed to load game class")
        else:
            logger.warning(f"BFS: game source not found for {s.game_id}")
            
    def _update_object_model(s, prev_raw, curr_raw, last_action_idx, last_action_data):
        """
        Maintains a provisional static/dynamic classification of objects.
        
        Objects are classified as STATIC (candidate targets) if they have not
        moved across multiple frames. However, if an action causes a previously
        static object to change (move, appear, disappear), it is immediately
        reclassified as DYNAMIC and removed from the target set.
        
        This means targets are always provisional — interaction can reveal
        that a 'static' object is actually responsive.
        """
        if not s._static_ready:
            s._frame_buffer.append(curr_raw.copy())
            if len(s._frame_buffer) >= 4:
                # Build initial static mask from first N frames
                base = s._frame_buffer[0]
                static = np.ones((64, 64), dtype=bool)
                for f in s._frame_buffer[1:]:
                    static &= (f == base)
                s._static_mask = static
                s._dynamic_mask = ~static
                s._static_ready = True
                
                cnt = np.bincount(curr_raw.flatten(), minlength=16)
                s._bg = int(cnt.argmax())
                
                # Identify structural colours (large static regions = play area border)
                cnt_static = np.bincount(curr_raw[s._static_mask].flatten(), minlength=16)
                cnt_static[s._bg] = 0
                structural_col = int(cnt_static.argmax())
                s._structural_colours = {structural_col} if cnt_static[structural_col] > 200 else set()
                
                # Initial target detection: rare static colours are candidate targets
                s._target_colours = set()
                for c in range(16):
                    if c == s._bg or c in s._structural_colours:
                        continue
                    n_static = int(np.sum(s._static_mask & (curr_raw == c)))
                    if 2 <= n_static <= 200:
                        s._target_colours.add(c)
                
                logger.info(f"Object model: bg={s._bg} structural={s._structural_colours} targets={s._target_colours}")

                # Detect goal groups by spatially clustering rare static pixels
                # Works regardless of where goals appear on screen
                from collections import defaultdict
                s._goal_groups = []
                rare_pixels = []
                for c in s._target_colours:
                    ys, xs = np.where(s._static_mask & (curr_raw == c))
                    for y, x in zip(ys, xs):
                        rare_pixels.append((int(x), int(y), c))

                if rare_pixels:
                    cluster_ids = list(range(len(rare_pixels)))

                    def find(i):
                        while cluster_ids[i] != i:
                            cluster_ids[i] = cluster_ids[cluster_ids[i]]
                            i = cluster_ids[i]
                        return i

                    def union(i, j):
                        ri, rj = find(i), find(j)
                        if ri != rj:
                            cluster_ids[ri] = rj

                    for i in range(len(rare_pixels)):
                        for j in range(i+1, len(rare_pixels)):
                            xi, yi, _ = rare_pixels[i]
                            xj, yj, _ = rare_pixels[j]
                            if abs(xi-xj) <= 12 and abs(yi-yj) <= 12:
                                union(i, j)

                    clusters = defaultdict(set)
                    for i, (x, y, c) in enumerate(rare_pixels):
                        clusters[find(i)].add(c)

                    s._goal_groups = [cols for cols in clusters.values()]
                    logger.info(f"Object model: detected {len(s._goal_groups)} goal groups: {s._goal_groups}")
            return

        # Already have a static mask — check if this action disturbed any static object
        diff = (prev_raw != curr_raw)
        if not np.any(diff):
            return

        # Check which previously-static colours changed
        disturbed = set()
        for c in s._target_colours | s._structural_colours:
            prev_static_pixels = s._static_mask & (prev_raw == c)
            if np.any(prev_static_pixels & diff):
                disturbed.add(c)

        if disturbed:
            # Reclassify disturbed colours as dynamic — they are NOT fixed targets
            for c in disturbed:
                s._target_colours.discard(c)
                # Update static mask to mark these pixels as dynamic
                s._static_mask[curr_raw == c] = False
                s._static_mask[prev_raw == c] = False
            s._dynamic_mask = ~s._static_mask
            logger.info(f"Object model: reclassified as dynamic after interaction: {disturbed}")

        # Also update static mask by removing any pixel that changed
        # This handles gradual revelation of dynamic objects
        s._static_mask[diff] = False
        s._dynamic_mask = ~s._static_mask
    def _try_bfs_solve(s, level_idx):
        """Try to solve current level. For L1+, uses A* with a goal
        heuristic derived from the previous level's win frame."""
        if s._bfs is None:
            return None

        prev_sol = s._bfs.solutions.get(level_idx - 1) if level_idx > 0 else None
        goal_heuristic = None

        # In _try_bfs_solve, replace the cumulative heuristic block with:
        if level_idx > 0 and prev_sol is not None:
            try:
                g = s._bfs.game_cls()
                g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                last_r = g.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                level_heuristics = []
        
                for pi in range(level_idx):
                    ps = s._bfs.solutions.get(pi)
                    if not ps:
                        break
                    f_level_init = np.array(last_r.frame[-1])
                    for act_id, data in ps:
                        ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                        last_r = g.perform_action(ai, raw=True)
                    f_level_win = np.array(last_r.frame[-1])
                    # Build heuristic once per level, reuse cached selectable actions
                    hfn = s._bfs._build_goal_heuristic(f_level_init, f_level_win)
                    level_heuristics.append((hfn, pi + 1))  # single replay, no re-instantiation
        
                if level_heuristics:
                    total_weight = sum(w for _, w in level_heuristics)
                    def goal_heuristic(f, game=None, _h=level_heuristics, _t=total_weight):
                        return sum(hfn(f, game) * w for hfn, w in _h) / _t

            except Exception as e:
                logger.warning(f"BFS L{level_idx}: goal heuristic failed: {e}")
                # Build demo model from prev level solution
                demo_model = None
                try:
                    g_demo = s._bfs.game_cls()
                    g_demo.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                    g_demo.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                    for pi in range(level_idx - 1):
                        ps = s._bfs.solutions.get(pi)
                        if not ps:
                            raise ValueError(f"missing L{pi}")
                        for act_id, data in ps:
                            ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                            g_demo.perform_action(ai, raw=True)
                    frames_and_actions = [(f_prev_init, None)]
                    for act_id, data in prev_sol:
                        ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                        r = g_demo.perform_action(ai, raw=True)
                        if r.frame:
                            frames_and_actions.append((np.array(r.frame[-1]), act_id))
                    demo_model = s._bfs._analyse_demo(frames_and_actions)
                except Exception as e:
                    logger.warning(f"BFS demo analysis failed: {e}")

                goal_heuristic_raw = s._bfs._build_goal_heuristic(f_prev_init, f_prev_win, demo_model=demo_model)
                
                # Calibrate: evaluate heuristic after one move to get baseline offset
                # L1 starts at L0 win state so raw h=0 there — we need relative change
                try:
                    g_cal = s._bfs.game_cls()
                    g_cal.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                    g_cal.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                    for pi in range(level_idx):
                        ps = s._bfs.solutions.get(pi)
                        if not ps: break
                        for act_id, data in ps:
                            ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                            g_cal.perform_action(ai, raw=True)
                    # Take one step to move away from L0 win state
                    r_cal = g_cal.perform_action(ActionInput(id=GameAction.ACTION1), raw=True)
                    if r_cal.frame:
                        f_after_move = np.array(r_cal.frame[-1])
                        h_after_move = goal_heuristic_raw(f_after_move, g_cal)
                        h_init = goal_heuristic_raw(f_prev_win, None)
                        logger.info(f"BFS L{level_idx}: heuristic calibration h_init={h_init:.2f} h_after_move={h_after_move:.2f}")
                        if h_after_move > h_init:
                            # Heuristic is working — use as-is
                            goal_heuristic = goal_heuristic_raw
                        else:
                            # Heuristic is flat — offset by subtracting init value
                            h_offset = h_init
                            def goal_heuristic(f, game=None, _offset=h_offset, _raw=goal_heuristic_raw):
                                return _raw(f, game) - _offset
                    else:
                        goal_heuristic = goal_heuristic_raw
                except Exception as e:
                    logger.warning(f"BFS heuristic calibration failed: {e}")
                    goal_heuristic = goal_heuristic_raw

        # Validate heuristic is not flat — if it is, replace with distance heuristic
        if goal_heuristic is not None:
            try:
                g_val = s._bfs.game_cls()
                g_val.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                last_r_val = g_val.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                for pi in range(level_idx):
                    ps = s._bfs.solutions.get(pi)
                    if not ps: break
                    for act_id, data in ps:
                        ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                        last_r_val = g_val.perform_action(ai, raw=True)
                if last_r_val.frame:
                    f_val = np.array(last_r_val.frame[-1])
                    h_vals = set()
                    h_vals.add(round(goal_heuristic(f_val, g_val), 4))
                    avail_val = [a for a in g_val._available_actions if 1 <= a <= 4]
                    for act_id in avail_val[:4]:
                        g2_val = copy.deepcopy(g_val)
                        r2_val = g2_val.perform_action(ActionInput(id=GameAction.from_id(act_id)), raw=True)
                        if r2_val.frame:
                            h_vals.add(round(goal_heuristic(np.array(r2_val.frame[-1]), g2_val), 4))
                    if len(h_vals) == 1 and level_idx in s._bfs.timed_out_levels:
                        logger.info(f"BFS L{level_idx}: heuristic is flat (h={list(h_vals)[0]}), switching to distance heuristic")
                        mover_colors, target_colors = s._bfs._probe_mover_target_colors(g_val)
                        if mover_colors and target_colors:
                            def goal_heuristic(f, game=None, _m=mover_colors, _t=target_colors):
                                centroids = {}
                                for c in range(16):
                                    mask = (f == c)
                                    n = int(np.sum(mask))
                                    if n < 2: continue
                                    ys, xs = np.where(mask)
                                    centroids[c] = (float(np.mean(xs)), float(np.mean(ys)))
                                targets = [(centroids[tc][0], centroids[tc][1]) for tc in _t if tc in centroids]
                                if not targets: return 0
                                total = 0
                                for mc in _m:
                                    if mc not in centroids: continue
                                    mx, my = centroids[mc]
                                    total += min(abs(mx - tx) + abs(my - ty) for tx, ty in targets)
                                return total
                            logger.info(f"BFS L{level_idx}: distance heuristic movers={mover_colors} targets={target_colors}")
            except Exception as e:
                logger.warning(f"BFS L{level_idx}: heuristic validation failed: {e}")
        
        sol = s._bfs.solve_level(level_idx, prev_solution=prev_sol, goal_heuristic=goal_heuristic)
        if sol:
            s._bfs_solution = sol
            s._bfs_step = 0
            return sol
        
        # First attempt failed — check if heuristic was flat and retry with distance heuristic
        if level_idx in s._bfs.timed_out_levels:
            try:
                g_val = s._bfs.game_cls()
                g_val.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                last_r_val = g_val.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                for pi in range(level_idx):
                    ps = s._bfs.solutions.get(pi)
                    if not ps: break
                    for act_id, data in ps:
                        ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                        last_r_val = g_val.perform_action(ai, raw=True)
                if last_r_val.frame:
                    f_val = np.array(last_r_val.frame[-1])
                    h_vals = set()
                    h_val_hfn = goal_heuristic if goal_heuristic is not None else (lambda f, game=None: 0)
                    h_vals.add(round(h_val_hfn(f_val, g_val), 4))
                    for act_id in [a for a in g_val._available_actions if 1 <= a <= 4][:4]:
                        g2_val = copy.deepcopy(g_val)
                        r2_val = g2_val.perform_action(ActionInput(id=GameAction.from_id(act_id)), raw=True)
                        if r2_val.frame:
                            h_vals.add(round(h_val_hfn(np.array(r2_val.frame[-1]), g2_val), 4))
                    if len(h_vals) == 1:
                        logger.info(f"BFS L{level_idx}: heuristic was flat — retrying with distance heuristic")
                        mover_colors, target_colors = s._bfs._probe_mover_target_colors(g_val)
                        if mover_colors and target_colors:
                            def dist_heuristic(f, game=None, _m=mover_colors, _t=target_colors):
                                centroids = {}
                                for c in range(16):
                                    mask = (f == c)
                                    n = int(np.sum(mask))
                                    if n < 2: continue
                                    ys, xs = np.where(mask)
                                    centroids[c] = (float(np.mean(xs)), float(np.mean(ys)))
                                targets = [(centroids[tc][0], centroids[tc][1]) for tc in _t if tc in centroids]
                                if not targets: return 0
                                total = 0
                                for mc in _m:
                                    if mc not in centroids: continue
                                    mx, my = centroids[mc]
                                    total += min(abs(mx - tx) + abs(my - ty) for tx, ty in targets)
                                return total
                            logger.info(f"BFS L{level_idx}: distance heuristic movers={mover_colors} targets={target_colors}")
                            sol = s._bfs.solve_level(level_idx, prev_solution=prev_sol, goal_heuristic=dist_heuristic)
                            if sol:
                                s._bfs_solution = sol
                                s._bfs_step = 0
                                return sol
            except Exception as e:
                logger.warning(f"BFS L{level_idx}: distance heuristic retry failed: {e}")
        
        return None
        return None

    def _tensor(s, fd):
        frame = s._raw(fd)
        oh=torch.zeros(16,64,64,dtype=torch.float32)
        oh.scatter_(0,torch.from_numpy(frame).unsqueeze(0),1)
        cnt=np.bincount(frame.flatten(),minlength=16)
        s._bg=int(cnt.argmax());mx=max(cnt.max(),1)
        bg_m=(frame==s._bg).astype(np.float32)
        rar=np.zeros((64,64),np.float32)
        for c in range(16):
            if cnt[c]>0:rar[frame==c]=1.0-cnt[c]/mx
        pad=np.pad(frame,1,mode='edge')
        edge=((frame!=pad[:-2,1:-1])|(frame!=pad[2:,1:-1])|(frame!=pad[1:-1,:-2])|(frame!=pad[1:-1,2:])).astype(np.float32)
        rp=np.linspace(0,1,64,dtype=np.float32).reshape(64,1).repeat(64,1)
        cp=np.linspace(0,1,64,dtype=np.float32).reshape(1,64).repeat(64,0)
        aug=torch.from_numpy(np.stack([bg_m,rar,edge,rp,cp]))
        d1=torch.zeros(3,64,64,dtype=torch.float32)
        for i,prev in enumerate(reversed(list(s.fhist))):
            if i>=3:break
            d1[i]=torch.from_numpy((frame!=prev).astype(np.float32))
        d2=torch.zeros(2,64,64,dtype=torch.float32)
        h=list(s.fhist)
        if len(h)>=2:d2[0]=torch.from_numpy((h[-1]!=h[-2]).astype(np.float32))
        if len(h)>=4:d2[1]=torch.from_numpy((h[-2]!=h[-4]).astype(np.float32))
        s.fhist.append(frame.copy())
        return torch.cat([oh,aug,d1,d2],0).to(s.device)

    def _detect_template(s, frame):
        mask=torch.ones(4096,dtype=torch.float32)
        col_act=np.sum(frame!=s._bg,axis=0)
        for c in range(20,44):
            if col_act[c]<=2 and np.sum(col_act[:c]>0)>=5 and np.sum(col_act[c+1:]>0)>=5:
                for y in range(64):
                    for x in range(c+1):mask[y*64+x]=0.05
                return mask
        row_act=np.sum(frame!=s._bg,axis=1)
        for r in range(20,44):
            if row_act[r]<=2 and np.sum(row_act[:r]>0)>=5 and np.sum(row_act[r+1:]>0)>=5:
                for y in range(r+1):
                    for x in range(64):mask[y*64+x]=0.05
                return mask
        return mask

    def _reward(s, prev_raw, curr_raw, prev_h, curr_h, last_action_idx=0, last_action_data=None):
        # Update object model with this transition
        s._update_object_model(prev_raw, curr_raw, last_action_idx, last_action_data)

        mask = np.ones((64,64), dtype=bool); mask[:2]=False; mask[62:]=False
        diff = (prev_raw != curr_raw) & mask
        changed = np.any(diff)
        r = 0.0

        if curr_h != prev_h:
            if curr_h not in s._visited_hashes:
                r += 1.5
                s._visited_hashes.add(curr_h)
            else:
                r += 0.2
        else:
            r -= 0.1

        if changed:
            r += 0.5

        smask = s._static_mask if s._static_ready else None
        curr_objs = fast_objects(curr_raw, s._bg, s._structural_colours, smask)
        prev_objs = s._prev_objs or []

        prev_colors = {o[0] for o in prev_objs}
        curr_colors = {o[0] for o in curr_objs}

        # Object movement reward
        if prev_objs and curr_objs:
            moved = 0
            for co in curr_objs:
                for po in prev_objs:
                    if co[0] == po[0]:
                        dist = abs(co[1]-po[1]) + abs(co[2]-po[2])
                        if 2 < dist < 20:
                            moved += 1
                            break
            if moved > 0:
                r += 0.3 * min(moved, 3)
                s._obj_moved = moved

            # Contact reward: dynamic object touching a target
            # Tracks progress per goal group and applies diminishing returns
            # to groups already ahead, forcing balanced multi-goal solving
            if s._static_ready and s._target_colours:
                group_progress = {}
                for dobj in curr_objs:
                    d_col, d_cx, d_cy, d_npix, d_w, d_h, d_x0, d_y0, d_x1, d_y1 = dobj
                    for tc in s._target_colours:
                        if tc == d_col:
                            continue
                        rs_ys, rs_xs = np.where(s._static_mask & (curr_raw == tc))
                        if len(rs_xs) == 0:
                            continue
                        rs_x0, rs_x1 = int(rs_xs.min()), int(rs_xs.max())
                        rs_y0, rs_y1 = int(rs_ys.min()), int(rs_ys.max())
                        x_gap = max(0, max(d_x0, rs_x0) - min(d_x1, rs_x1))
                        y_gap = max(0, max(d_y0, rs_y0) - min(d_y1, rs_y1))
                        contact_score = 0.0
                        if x_gap <= 2 and y_gap <= 2:
                            contact_score = 2.0
                        elif x_gap <= 10 and y_gap <= 10:
                            contact_score = 0.5
                        if contact_score > 0:
                            group_idx = None
                            for gi, grp in enumerate(s._goal_groups):
                                if tc in grp:
                                    group_idx = gi
                                    break
                            if group_idx is not None:
                                group_progress[group_idx] = max(
                                    group_progress.get(group_idx, 0.0),
                                    contact_score)
                            else:
                                r += contact_score

                if group_progress and s._goal_groups:
                    scores = [group_progress.get(i, 0.0) for i in range(len(s._goal_groups))]
                    for gi, score in enumerate(scores):
                        if score > 0:
                            other_scores = [sc for j, sc in enumerate(scores) if j != gi]
                            max_other = max(other_scores) if other_scores else 0.0
                            lag_bonus = 1.0 if score <= max_other else 0.5
                            r += score * lag_bonus
                elif group_progress:
                    for score in group_progress.values():
                        r += score

            # Composite object movement toward targets
            if s._static_ready and s._target_colours:
                prev_composites = find_composite_objects(prev_objs)
                curr_composites = find_composite_objects(curr_objs)
                for cc in curr_composites:
                    cc_cols = {o[0] for o in cc}
                    cc_cx = float(np.mean([o[1] for o in cc]))
                    cc_cy = float(np.mean([o[2] for o in cc]))
                    # Find nearest target
                    best_target_dist = 999.0
                    for tc in s._target_colours:
                        rs_ys, rs_xs = np.where(s._static_mask & (curr_raw == tc))
                        if len(rs_xs) == 0:
                            continue
                        td = abs(float(np.mean(rs_xs)) - cc_cx) + abs(float(np.mean(rs_ys)) - cc_cy)
                        best_target_dist = min(best_target_dist, td)
                    # Compare to previous position of same composite
                    for pc in prev_composites:
                        pc_cols = {o[0] for o in pc}
                        if cc_cols == pc_cols:
                            pc_cx = float(np.mean([o[1] for o in pc]))
                            pc_cy = float(np.mean([o[2] for o in pc]))
                            # Reward moving toward target
                            prev_target_dist = 999.0
                            for tc in s._target_colours:
                                rs_ys, rs_xs = np.where(s._static_mask & (curr_raw == tc))
                                if len(rs_xs) == 0:
                                    continue
                                td = abs(float(np.mean(rs_xs)) - pc_cx) + abs(float(np.mean(rs_ys)) - pc_cy)
                                prev_target_dist = min(prev_target_dist, td)
                            if prev_target_dist - best_target_dist > 1:
                                r += 0.4  # moved closer to a target
                            break

        # Disappeared object reward (pickup / elimination)
        disappeared = prev_colors - curr_colors
        if disappeared:
            r += 2.0 * len(disappeared)

        s._prev_objs = curr_objs
        return r

    def _sample(s, logits, avail=None, temp=1.0):
        al=logits[:5].clone();cl=logits[5:5+4096].clone()
        if avail is not None and len(avail)>0:
            mask=torch.full_like(al,float('-inf'));a6=False
            for a in avail:
                aid=a.value if hasattr(a,'value') else int(a)
                if 1<=aid<=5:mask[aid-1]=0.0
                elif aid==6:a6=True
            al=al+mask
            if not a6:cl=cl+torch.full_like(cl,float('-inf'))
        if s._wm is not None:cl=cl+torch.log(s._wm.to(s.device).clamp(min=0.01))
        # Offline strategy-prior training: small action bias from dataset-derived ARC hints.
        # BFS solutions and live evidence still dominate; this only shapes fallback exploration.
        try:
            if getattr(s, "_strategy_prior", None):
                bias=torch.tensor(getattr(s,"_prior_action_bias",[0,0,0,0,0]),dtype=al.dtype,device=s.device)
                al=al+(bias*float(getattr(s,"_prior_strength",0.35)))
                cl=cl+(float(getattr(s,"_prior_click_bias",0.0))*float(getattr(s,"_prior_strength",0.35)))
        except Exception:
            pass
        ap=torch.sigmoid(al/temp);cp=torch.sigmoid(cl/temp)/(s.G*s.G)
        allp=torch.cat([ap,cp]);sm=allp.sum()
        if sm<1e-8:allp=torch.ones_like(allp)/len(allp)
        else:allp=allp/sm
        idx=np.random.choice(len(allp),p=allp.cpu().numpy())
        if idx<5:return idx,None
        ci=idx-5;return 5,(ci//s.G,ci%s.G)

    def _heuristic(s, frame, avail, step):
        av=set(int(a.value) if hasattr(a,'value') else int(a) for a in avail)
        # Training-prior opening curriculum. It only runs during the existing
        # heuristic warmup window and only for actions currently exposed by the env.
        try:
            if getattr(s, "_strategy_prior", None) and step < len(getattr(s, "_prior_opening", [])):
                d=int(s._prior_opening[step])
                if d in av:
                    return d-1,None
        except Exception:
            pass
        for d in[1,2,3,4]:
            if d in av and step<4:return d-1,None
        if 6 in av:
            cnt=np.bincount(frame.flatten(),minlength=16);targets=[]
            for c in range(16):
                if c==s._bg or cnt[c]==0 or cnt[c]>2000:continue
                ys,xs=np.where(frame==c)
                if len(ys)>=2:targets.append((int(np.median(xs)),int(np.median(ys)),len(ys)))
            targets.sort(key=lambda t:t[2]);pidx=step-4
            if 0<=pidx<len(targets):return 5,(targets[pidx][1],targets[pidx][0])
        if 5 in av:return 4,None
        choices=[a for a in av if 1<=a<=5]
        if choices:return random.choice(choices)-1,None
        return 0,None

    def _frame_to_tensor(s, frame):
        oh=torch.zeros(16,64,64,dtype=torch.float32)
        oh.scatter_(0,torch.from_numpy(frame).unsqueeze(0),1)
        cnt=np.bincount(frame.flatten(),minlength=16)
        bg=int(cnt.argmax());mx=max(cnt.max(),1)
        bg_m=(frame==bg).astype(np.float32)
        rar=np.zeros((64,64),np.float32)
        for c in range(16):
            if cnt[c]>0:rar[frame==c]=1.0-cnt[c]/mx
        pad=np.pad(frame,1,mode='edge')
        edge=((frame!=pad[:-2,1:-1])|(frame!=pad[2:,1:-1])|(frame!=pad[1:-1,:-2])|(frame!=pad[1:-1,2:])).astype(np.float32)
        rp=np.linspace(0,1,64,dtype=np.float32).reshape(64,1).repeat(64,1)
        cp=np.linspace(0,1,64,dtype=np.float32).reshape(1,64).repeat(64,0)
        aug=torch.from_numpy(np.stack([bg_m,rar,edge,rp,cp]))
        zeros=torch.zeros(5,64,64,dtype=torch.float32)
        return torch.cat([oh,aug,zeros],0)

    def _train(s):
        if len(s.buf)<s.bsz:return
        indices=np.random.choice(len(s.buf),s.bsz,replace=False)
        batch=[s.buf[i] for i in indices]
        states=torch.stack([s._frame_to_tensor(e['s']).to(s.device) for e in batch])
        acts=torch.tensor([e['a'] for e in batch],dtype=torch.long,device=s.device)
        rews=torch.tensor([e['r'] for e in batch],dtype=torch.float32,device=s.device)
        rews=torch.sigmoid(rews);s.opt.zero_grad()
        logits=s.net(states)
        acts_c=acts.clamp(0,logits.size(1)-1)
        sel=logits.gather(1,acts_c.unsqueeze(1)).squeeze(1)
        loss=F.binary_cross_entropy_with_logits(sel,rews)
        p=torch.sigmoid(logits);loss=loss-0.0001*p[:,:5].mean()-0.00001*p[:,5:].mean()
        loss.backward();s.opt.step()

    def _get_aem_tensors(s):
        if len(s._aem_diffs)<2:return None,None,None
        M=len(s._aem_diffs)
        diffs=torch.zeros(1,M,1,64,64,device=s.device)
        acts=torch.zeros(1,M,dtype=torch.long,device=s.device)
        rews=torch.zeros(1,M,device=s.device)
        for i,(d,a,r) in enumerate(zip(s._aem_diffs,s._aem_actions,s._aem_rewards)):
            diffs[0,i,0]=torch.from_numpy(d.astype(np.float32));acts[0,i]=min(a,4);rews[0,i]=r
        return diffs,acts,rews

    def is_done(s, frames, lf):
        try: return lf.state is GameState.WIN or (time.time()-s.start_time) >= 8*3600-300
        except: return True

    def choose_action(s, frames, lf):
        try:
            lvl = s._lvl(lf)

            # ===== LEVEL CHANGE =====
            if lvl != s.cl:
                # Init BFS solver on first level
                if not s._bfs_tried:
                    s._bfs_tried = True
                    s._init_bfs()

                # Try BFS for this level
                s._bfs_solution = None
                s._bfs_step = 0
                if s._bfs:
                    s._try_bfs_solve(lvl)

                # Init CNN fallback
                s.buf.clear(); s.buf_h.clear()
                s.net = ForgeNet(s.IN, s.G).to(s.device)
                for wp in ['/kaggle/input/forge-pretrained-weights/pretrained_weights.pt',
                           'pretrained_weights.pt']:
                    try:
                        if os.path.exists(wp):
                            state=torch.load(wp,map_location=s.device,weights_only=True)
                            ms=s.net.state_dict()
                            for k in list(state.keys()):
                                if k in ms and state[k].shape==ms[k].shape:ms[k]=state[k]
                            s.net.load_state_dict(ms);break
                    except: pass
                s.opt = optim.Adam(s.net.parameters(), lr=0.0003)
                s.pt=None;s.pai=None;s.pr=None;s.ph=None
                s.cl=lvl;s.fhist.clear();s.la=0
                s._wd=False;s._wm=None
                s._aem_diffs.clear();s._aem_actions.clear();s._aem_rewards.clear()
                s._prev_objs=None;s._obj_moved=0;s._ckpt_hash=None;s._unproductive=0
                # FIX 1: Reset visited hashes on every level change
                s._visited_hashes = set()
                # Reset object model
                s._frame_buffer = []
                s._static_mask = None
                s._dynamic_mask = None
                s._static_ready = False
                s._structural_colours = set()
                s._target_colours = set()
                s._goal_groups = []
                # Strategy-prior replay warm-start. This is deliberately light: it
                # initializes the fallback network toward the dataset action prior
                # but does not override verified BFS or live reward.
                try:
                    if getattr(s, "_strategy_prior", None) and len(s.buf) < s.bsz:
                        dummy = np.zeros((64,64), dtype=np.int64)
                        for ai, b in enumerate(getattr(s, "_prior_action_bias", [0,0,0,0,0])):
                            reps = max(1, int(2 + abs(float(b))*4))
                            rew = 0.55 + max(-0.2, min(0.2, float(b)))
                            for _ in range(reps):
                                s.buf.append({'s':dummy.copy(),'a':ai,'r':rew})
                except Exception:
                    pass
                # FIX 4: Only reset epsilon if BFS didn't solve this level.
                # If BFS solved it, keep current eps so CNN fallback (if needed)
                # benefits from accumulated exploration knowledge.
                if not s._bfs_solution:
                    s._eps = 0.15

                # CLTI — inject BFS demos from previous level into CNN replay buffer
                # FIX 2: Use perform_action frame[-1] consistently with _raw(),
                # instead of get_pixels() which returns a different format.
                if lvl > 0 and s._bfs and s._bfs.solutions.get(lvl - 1):
                    prev_sol = s._bfs.solutions[lvl - 1]
                    try:
                        replay_game = s._bfs.game_cls()
                        replay_game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                        r0 = replay_game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
                        if r0.frame:
                            # Start from the post-reset frame, consistent with _raw()
                            prev_frame = np.array(r0.frame[-1], dtype=np.int64)
                            for act_id, data in prev_sol:
                                ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                                result = replay_game.perform_action(ai, raw=True)
                                action_idx = (act_id - 1) if act_id <= 5 else (
                                    5 + data.get('y', 0) * 64 + data.get('x', 0) if data else 0)
                                s.buf.append({'s': prev_frame.copy(), 'a': action_idx, 'r': 2.0})
                                # Advance prev_frame using the action result, not get_pixels()
                                if result.frame:
                                    prev_frame = np.array(result.frame[-1], dtype=np.int64)
                            if len(s.buf) >= s.bsz:
                                for _ in range(min(20, len(s.buf) // s.bsz)):
                                    s._train()
                                logger.info(f"CLTI: injected {len(prev_sol)} expert demos from L{lvl-1}")
                    except Exception as e:
                        logger.warning(f"CLTI failed: {e}")

            # ===== RESET =====
            if lf.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
                s.pt=None;s.pai=None;s.pr=None;s.ph=None
                return GameAction.RESET

            # ===== BFS SOLUTION EXECUTION =====
            if s._bfs_solution and s._bfs_step < len(s._bfs_solution):
                act_id, data = s._bfs_solution[s._bfs_step]
                s._bfs_step += 1
                sel = GameAction.from_id(act_id)
                clean_data = {k: v for k, v in data.items() if k != 'game_id'} if isinstance(data, dict) else data
                s._last_action_data = clean_data if clean_data else None
                if clean_data:
                    try:
                        sel.set_data(clean_data)
                    except:
                        try:
                            sel.data = clean_data
                        except:
                            pass
                raw = s._raw(lf)
                s.fhist.append(raw.copy())
                s.pr = raw.copy()
                s.la += 1
                return sel

            # ===== TERMUX TORCHLESS FALLBACK =====
            # If torch.nn is unavailable, avoid CNN tensor path entirely.
            # Use deterministic heuristic + offline strategy-prior opening.
            if not FORGE_TORCH_AVAILABLE:
                raw = s._raw(lf)
                avail = getattr(lf, 'available_actions', None) or []

                # Prefer trained opening curriculum for the first few moves.
                prior_open = getattr(s, "_prior_opening", []) or []
                picked = None
                if s.la < len(prior_open):
                    wanted = int(prior_open[s.la])
                    for a in avail:
                        aid = a.value if hasattr(a, "value") else int(a)
                        if aid == wanted:
                            picked = a
                            break

                if picked is None:
                    try:
                        aidx, coords = s._heuristic(raw, avail, s.la)
                    except Exception:
                        aidx, coords = 0, None

                    if aidx < 5:
                        picked = s.al[aidx]
                    else:
                        picked = GameAction.ACTION6
                        if coords:
                            y, x = coords
                            click_data = {"x": int(x), "y": int(y)}
                            s._last_action_data = click_data
                            try:
                                picked.set_data(click_data)
                            except Exception:
                                try:
                                    picked.data = click_data
                                except Exception:
                                    pass

                s.pr = raw.copy()
                s.ph = hashlib.md5(raw.tobytes()).hexdigest()[:16]
                s.la += 1
                try:
                    picked.reasoning = "termux_torchless_prior_heuristic"
                except Exception:
                    pass
                return picked

            # ===== CNN FALLBACK =====
            tensor = s._tensor(lf)
            raw = s._raw(lf)
            ch = hashlib.md5(raw.tobytes()).hexdigest()[:16]
            avail = getattr(lf, 'available_actions', None) or []
            s._undo_avail = any((a.value if hasattr(a,'value') else int(a))==7 for a in avail)

            if s.pt is not None and s.pai is not None:
                mask=np.ones((64,64),dtype=bool);mask[:2]=False;mask[62:]=False
                diff_map=(s.pr!=raw)&mask;changed=np.any(diff_map)
                eh=hashlib.md5(s.pr.tobytes()[:1000]+str(s.pai).encode()).hexdigest()[:16]
                if eh not in s.buf_h:
                    r=s._reward(s.pr, raw, '', ch, s.pai, getattr(s, '_last_action_data', None))
                    s.buf.append({'s':s.pr.copy(),'a':s.pai,'r':r})
                    s.buf_h.add(eh)
                    if changed:
                        s._aem_diffs.append(diff_map)
                        s._aem_actions.append(min(s.pai,4))
                        s._aem_rewards.append(r)
                if changed:s._ckpt_hash=ch;s._unproductive=0
                else:s._unproductive+=1

            avail_idx=[]
            for a in avail:
                aid=a.value if hasattr(a,'value') else int(a)
                if 1<=aid<=5:avail_idx.append(aid-1)
                elif aid==6:avail_idx.extend([5+i for i in range(0,4096,128)])

            if s._wm is None:s._wm=s._detect_template(raw)

            if s._undo_avail and s._unproductive>=30 and s._ckpt_hash:
                s._unproductive=0;a=GameAction.ACTION7;a.reasoning="undo"
                s.pt=tensor;s.pai=6;s.pr=raw.copy();s.ph=ch;s.la+=1;return a

            if not s._wd:
                if s.la<10:aidx,coords=s._heuristic(raw,avail,s.la)
                else:
                    s._wd=True
                    for _ in range(min(5,len(s.buf)//s.bsz)):s._train()

            if s._wd:
                if random.random()<s._eps:
                    aidx,coords=s._sample(torch.zeros(4101,device=s.device),avail,temp=2.0)
                else:
                    with torch.no_grad():
                        mem=s._get_aem_tensors()
                        if mem[0] is not None:logits=s.net(tensor.unsqueeze(0),*mem).squeeze(0)
                        else:logits=s.net(tensor.unsqueeze(0)).squeeze(0)
                    aidx,coords=s._sample(logits,avail,temp=0.5)
                s._eps=max(s._eps_min,s._eps*s._eps_decay)
            elif s.la>=10:s._wd=True;aidx,coords=0,None

            if aidx<5:
                sel=s.al[aidx]
            else:
                sel=GameAction.ACTION6;y,x=coords
                click_data={"x":int(x),"y":int(y)}
                s._last_action_data=click_data
                # Critical: ACTION6 is coordinate-bearing. Emit the data, not just
                # a side-channel memory entry, so the environment receives the click.
                try:
                    sel.set_data(click_data)
                except Exception:
                    try:
                        sel.data=click_data
                    except Exception:
                        pass

            s.pt=tensor;s.pai=aidx if aidx<5 else(5+coords[0]*s.G+coords[1])
            s.pr=raw.copy();s.ph=ch;s.la+=1
            if s.action_counter%s.tfreq==0 and s._wd:s._train()
            return sel

        except Exception as e:
            traceback.print_exc()
            a=random.choice(s.al);a.reasoning=f"err:{e}";return a

# =====================================================================
# GLYPHMATICS AGENT 5 FUSION LAYER
# Appended after Ash base. This preserves Ash behavior unless a helper
# symbol is explicitly used by the existing agent.
# =====================================================================



# =====================================================================
# SAFE COMPETITION FUSION GUARD
# No OpenAI/API/network calls are used at runtime.
# Ash base remains primary execution path.
# =====================================================================


# === SAFE SOLUTION PRESERVATION GUARD ===
# Previous experimental action compression was disabled for competition use.
# ARC solutions often require repeated moves or deliberate A-B-A oscillations;
# blind compression can invalidate a solved replay. Keep the validated BFS path
# byte-for-byte unless a future compressor performs full replay validation.
def _compress_actions(seq):
    return seq



# =====================================================================
# FINAL STATE SIMILARITY PRUNING LAYER
# Reduces near-duplicate BFS states without changing main agent structure.
# =====================================================================

def _sim_signature(frame, block=4):
    import numpy as np, hashlib
    f = np.asarray(frame)
    if f.ndim != 2:
        return hashlib.md5(f.tobytes()).hexdigest()[:16]

    h, w = f.shape
    h2 = h - (h % block)
    w2 = w - (w % block)
    f = f[:h2, :w2]

    # coarse mode-like signature using block mean rounded.
    small = f.reshape(h2 // block, block, w2 // block, block).mean(axis=(1, 3))
    small = np.rint(small).astype("uint8")
    return hashlib.md5(small.tobytes()).hexdigest()[:16]


def _install_similarity_pruning():
    try:
        orig = BFSSolver._perform_and_drain
    except Exception:
        return False

    def wrapped(self, game, ai, max_drain=5, drain=True):
        r = orig(self, game, ai, max_drain=max_drain, drain=drain)

        try:
            if not hasattr(self, "_sim_seen"):
                self._sim_seen = set()

            if getattr(r, "frame", None):
                sig = _sim_signature(r.frame[-1], block=4)

                # Tag result with similarity marker for BFS V5 if available.
                setattr(r, "_sim_signature", sig)

                # Do not mutate winning frames; only mark duplicates.
                if sig in self._sim_seen:
                    setattr(r, "_sim_duplicate", True)
                else:
                    self._sim_seen.add(sig)
                    setattr(r, "_sim_duplicate", False)
        except Exception:
            pass

        return r

    BFSSolver._perform_and_drain = wrapped
    return True


try:
    if _install_similarity_pruning():
        print("[OK] State similarity pruning active")
except Exception as e:
    print("[ERR similarity pruning]", e)

# =====================================================================



# =====================================================================
# FORGE v19.5 INLINE GAMEPLAY LISTER
# Inline run-log instrumentation for level starts, available actions,
# chosen interactions, reward deltas, BFS routes, transfer attempts,
# hidden/transient fields, wins, losses, and better-route candidates.
#
# Safety:
# - observes only; does not change solver scoring/ranking/action selection
# - no network/API/file dependency
# - bounded by env limits
# =====================================================================

class GameplayLister:
    def __init__(self, prefix="RUN"):
        self.prefix = str(prefix)
        self.enabled = os.environ.get("FORGE_GAMEPLAY_LOG", "1").lower() not in ("0", "false", "no", "off")
        self.trace = int(os.environ.get("FORGE_GAMEPLAY_TRACE", "2"))
        self.route_limit = int(os.environ.get("FORGE_GAMEPLAY_ROUTE_LIMIT", "256"))
        self.action_limit = int(os.environ.get("FORGE_GAMEPLAY_ACTION_LIMIT", "128"))
        self.step_limit = int(os.environ.get("FORGE_GAMEPLAY_STEP_LIMIT", "1000"))
        self.best_route_len = {}
        self.step_counts = {}
        self.last_state = {}

    def _safe(self, value, max_len=1200):
        try:
            text = str(value)
        except Exception:
            text = "<unprintable>"
        text = text.replace("\n", " ").replace("\r", " ")
        if len(text) > max_len:
            return text[:max_len - 3] + "..."
        return text

    def emit(self, level, tag, msg, min_trace=1):
        if not self.enabled or self.trace < min_trace:
            return
        try:
            print(f"[GLIST][{self.prefix}][L{level}][{tag}] {self._safe(msg)}", flush=True)
        except Exception:
            pass

    def frame_stats(self, frame):
        try:
            arr = np.asarray(frame)
            if arr.ndim == 3:
                arr = arr[-1]
            if arr.size == 0:
                return "frame=empty"
            vals, counts = np.unique(arr, return_counts=True)
            pairs = sorted([(int(v), int(c)) for v, c in zip(vals, counts)], key=lambda x: (-x[1], x[0]))
            bg, bg_count = pairs[0]
            non_bg = int(arr.size - bg_count)
            colors = ",".join(f"{v}:{c}" for v, c in pairs[:8])
            sig = hashlib.md5(arr.astype("uint8", copy=False).tobytes()).hexdigest()[:12]
            mask = arr != bg
            if np.any(mask):
                ys, xs = np.where(mask)
                bbox = f"bbox=({int(xs.min())},{int(ys.min())})-({int(xs.max())},{int(ys.max())})"
            else:
                bbox = "bbox=none"
            return f"sig={sig} shape={tuple(arr.shape)} bg={bg} non_bg={non_bg} colors={colors} {bbox}"
        except Exception as e:
            return f"frame_stats_error={type(e).__name__}:{e}"

    def _act_name(self, act):
        try:
            if hasattr(act, "name"):
                return act.name
            if hasattr(act, "value"):
                return f"ACTION{int(act.value)}"
            return f"ACTION{int(act)}"
        except Exception:
            return self._safe(act, 80)

    def _act_data(self, act):
        try:
            data = getattr(act, "data", None)
            if data is None and hasattr(act, "_data"):
                data = getattr(act, "_data", None)
            if data is None:
                return None
            try:
                return dict(data)
            except Exception:
                return data
        except Exception:
            return None

    def fmt_action(self, item):
        try:
            if isinstance(item, tuple):
                act, data = item
            else:
                act, data = item, self._act_data(item)
            name = self._act_name(act)
            if isinstance(data, dict) and ("x" in data or "y" in data):
                return f"{name}(x={int(data.get('x', -1))},y={int(data.get('y', -1))})"
            if data:
                return f"{name}({self._safe(data, 100)})"
            return name
        except Exception as e:
            return f"ACTION_FMT_ERROR:{type(e).__name__}:{e}"

    def fmt_actions(self, actions, limit=None):
        try:
            if actions is None:
                return "[]"
            items = list(actions)
            limit = self.action_limit if limit is None else int(limit)
            shown = [self.fmt_action(a) for a in items[:limit]]
            if len(items) > limit:
                shown.append(f"... +{len(items) - limit} more")
            return "[" + ", ".join(shown) + "]"
        except Exception as e:
            return f"actions_fmt_error={type(e).__name__}:{e}"

    def fmt_route(self, route):
        try:
            if route is None:
                return "none"
            items = list(route)
            shown = [self.fmt_action(a) for a in items[:self.route_limit]]
            if len(items) > self.route_limit:
                shown.append(f"... +{len(items) - self.route_limit} more")
            return " -> ".join(shown)
        except Exception as e:
            return f"route_fmt_error={type(e).__name__}:{e}"

    def level_start(self, level, frame=None, available=None, state=None, source="env"):
        self.step_counts[level] = 0
        msg = f"START source={source}"
        if state is not None:
            msg += f" state={self._safe(state, 80)}"
        if available is not None:
            msg += f" available={self.fmt_actions([(a, None) for a in list(available)], limit=64)}"
        if frame is not None:
            msg += f" | {self.frame_stats(frame)}"
        self.emit(level, "LEVEL", msg, min_trace=1)

    def state_event(self, level, state, frame=None):
        state_s = self._safe(state, 80)
        prev = self.last_state.get(level)
        if prev != state_s:
            self.last_state[level] = state_s
            msg = f"state={state_s}"
            if frame is not None and self.trace >= 3:
                msg += f" | {self.frame_stats(frame)}"
            if "WIN" in state_s:
                self.emit(level, "WIN", msg, min_trace=1)
            elif "GAME_OVER" in state_s or "LOSE" in state_s or "LOSS" in state_s:
                self.emit(level, "LOSS", msg, min_trace=1)
            else:
                self.emit(level, "STATE", msg, min_trace=3)

    def runtime_action(self, level, step, source, action, frame=None, available=None, state=None):
        count = self.step_counts.get(level, 0)
        if count >= self.step_limit:
            if count == self.step_limit:
                self.emit(level, "STEP", f"step_log_limit={self.step_limit}; suppressing further action lines", min_trace=1)
                self.step_counts[level] = count + 1
            return
        self.step_counts[level] = count + 1
        msg = f"step={step} source={source} action={self.fmt_action(action)}"
        if state is not None:
            msg += f" state={self._safe(state, 80)}"
        if available is not None and self.trace >= 3:
            msg += f" available={self.fmt_actions([(a, None) for a in list(available)], limit=64)}"
        if frame is not None and self.trace >= 3:
            msg += f" | {self.frame_stats(frame)}"
        self.emit(level, "STEP", msg, min_trace=1)

    def reward(self, level, step, action_idx, action_data, reward_value, changed_px=None, prev_frame=None, curr_frame=None):
        msg = f"after_step={step} prev_action={self.fmt_action((action_idx, action_data))} reward={float(reward_value):.3f}"
        if changed_px is not None:
            msg += f" changed_px={int(changed_px)}"
        if curr_frame is not None and self.trace >= 3:
            msg += f" | curr={self.frame_stats(curr_frame)}"
        self.emit(level, "REWARD", msg, min_trace=2)

    def scan_start(self, level, frame, bg, available):
        msg = f"scan_start bg={bg} available={self.fmt_actions([(a, None) for a in list(available or [])], limit=64)}"
        if frame is not None:
            msg += f" | {self.frame_stats(frame)}"
        self.emit(level, "SCAN", msg, min_trace=2)

    def scan_done(self, level, actions):
        self.emit(level, "SCAN", f"scan_done count={len(actions) if actions is not None else 0} actions={self.fmt_actions(actions)}", min_trace=2)

    def route(self, level, source, route, elapsed=None, status="candidate"):
        try:
            ln = len(route) if route is not None else 0
        except Exception:
            ln = -1
        prev = self.best_route_len.get(level)
        if route is not None:
            if prev is None:
                better = "new_best"
                self.best_route_len[level] = ln
            elif ln >= 0 and ln < prev:
                better = f"better_by={prev - ln}"
                self.best_route_len[level] = ln
            elif ln == prev:
                better = "ties_best"
            else:
                better = f"longer_by={ln - prev}"
        else:
            better = "none"
        msg = f"{status} source={source} len={ln} best={better}"
        if elapsed is not None:
            msg += f" elapsed={elapsed:.2f}s"
        msg += f" route={self.fmt_route(route)}"
        self.emit(level, "ROUTE", msg, min_trace=1)

    def result(self, level, method, route=None, elapsed=None, reason=""):
        if route:
            self.route(level, method, route, elapsed=elapsed, status="SOLVED")
        else:
            msg = f"FAILED source={method}"
            if elapsed is not None:
                msg += f" elapsed={elapsed:.2f}s"
            if reason:
                msg += f" reason={reason}"
            self.emit(level, "RESULT", msg, min_trace=1)

    def fields(self, level, kind, fields):
        self.emit(level, "FIELDS", f"{kind}={self._safe(fields, 800)}", min_trace=2)

    def exception(self, level, where, err):
        self.emit(level, "ERROR", f"{where}: {type(err).__name__}:{err}", min_trace=1)


def _forge_level_from_frame(agent, lf):
    try:
        return agent._lvl(lf)
    except Exception:
        try:
            return getattr(lf, "score", None) or getattr(lf, "levels_completed", "?")
        except Exception:
            return "?"

def _forge_raw_from_frame(agent, lf):
    try:
        return agent._raw(lf)
    except Exception:
        try:
            return np.asarray(getattr(lf, "frame", None))[-1]
        except Exception:
            return None

def _install_inline_gameplay_lister():
    try:
        if getattr(MyAgent, "_forge_glist_installed", False):
            return False

        orig_agent_init = MyAgent.__init__
        def agent_init_wrapped(self, *a, **kw):
            orig_agent_init(self, *a, **kw)
            try:
                self._glog = GameplayLister(prefix="RUN")
                self._glog_last_level = None
                self._glog_last_action = None
            except Exception:
                pass
        MyAgent.__init__ = agent_init_wrapped

        orig_choose = MyAgent.choose_action
        def choose_action_wrapped(self, frames, lf):
            lvl = _forge_level_from_frame(self, lf)
            raw = _forge_raw_from_frame(self, lf)
            avail = getattr(lf, "available_actions", None) or []
            state = getattr(lf, "state", None)
            try:
                if not hasattr(self, "_glog"):
                    self._glog = GameplayLister(prefix="RUN")
                if getattr(self, "_glog_last_level", None) != lvl:
                    self._glog.level_start(lvl, frame=raw, available=avail, state=state, source="environment")
                    self._glog_last_level = lvl
                self._glog.state_event(lvl, state, frame=raw)
            except Exception:
                pass

            try:
                action = orig_choose(self, frames, lf)
            except Exception as e:
                try:
                    self._glog.exception(lvl, "choose_action", e)
                except Exception:
                    pass
                raise

            try:
                source = "bfs-replay" if getattr(self, "_bfs_solution", None) and getattr(self, "_bfs_step", 0) > 0 and getattr(self, "_bfs_step", 0) <= len(getattr(self, "_bfs_solution", [])) else "policy"
                step = getattr(self, "la", "?")
                self._glog.runtime_action(lvl, step, source, action, frame=raw, available=avail, state=state)
                self._glog_last_action = action
            except Exception:
                pass
            return action
        MyAgent.choose_action = choose_action_wrapped

        orig_reward = MyAgent._reward
        def reward_wrapped(self, prev_raw, curr_raw, prev_h, curr_h, last_action_idx=0, last_action_data=None):
            reward_value = orig_reward(self, prev_raw, curr_raw, prev_h, curr_h, last_action_idx, last_action_data)
            try:
                changed_px = int(np.sum(np.asarray(prev_raw) != np.asarray(curr_raw)))
                lvl = getattr(self, "cl", "?")
                if hasattr(self, "_glog"):
                    self._glog.reward(lvl, getattr(self, "la", "?"), last_action_idx, last_action_data, reward_value, changed_px=changed_px, prev_frame=prev_raw, curr_frame=curr_raw)
            except Exception:
                pass
            return reward_value
        MyAgent._reward = reward_wrapped

        orig_is_done = MyAgent.is_done
        def is_done_wrapped(self, frames, lf):
            done = orig_is_done(self, frames, lf)
            try:
                lvl = _forge_level_from_frame(self, lf)
                raw = _forge_raw_from_frame(self, lf)
                state = getattr(lf, "state", None)
                if not hasattr(self, "_glog"):
                    self._glog = GameplayLister(prefix="RUN")
                if done:
                    tag = "done"
                    if str(state).find("WIN") >= 0:
                        tag = "win"
                    elif str(state).find("GAME_OVER") >= 0:
                        tag = "loss"
                    self._glog.emit(lvl, "DONE", f"{tag} state={state} | {self._glog.frame_stats(raw) if raw is not None else 'no_frame'}", min_trace=1)
            except Exception:
                pass
            return done
        MyAgent.is_done = is_done_wrapped

        orig_try_bfs = MyAgent._try_bfs_solve
        def try_bfs_wrapped(self, lvl):
            t0 = time.time()
            result = orig_try_bfs(self, lvl)
            try:
                if not hasattr(self, "_glog"):
                    self._glog = GameplayLister(prefix="RUN")
                sol = getattr(self, "_bfs_solution", None)
                if sol:
                    self._glog.route(lvl, "bfs-selected", sol, elapsed=time.time() - t0, status="candidate")
                else:
                    self._glog.result(lvl, "bfs-selected", None, elapsed=time.time() - t0, reason="no_route_selected")
            except Exception:
                pass
            return result
        MyAgent._try_bfs_solve = try_bfs_wrapped

        orig_bfs_init = BFSSolver.__init__
        def bfs_init_wrapped(self, *a, **kw):
            orig_bfs_init(self, *a, **kw)
            try:
                self.lister = GameplayLister(prefix="BFS")
                self._glist_active_level = "?"
            except Exception:
                pass
        BFSSolver.__init__ = bfs_init_wrapped

        orig_scan = BFSSolver._scan_actions
        def scan_wrapped(self, game, f0, bg):
            lvl = getattr(self, "_glist_active_level", "?")
            try:
                if not hasattr(self, "lister"):
                    self.lister = GameplayLister(prefix="BFS")
                self.lister.scan_start(lvl, f0, bg, getattr(game, "_available_actions", None))
            except Exception:
                pass
            actions = orig_scan(self, game, f0, bg)
            try:
                self.lister.scan_done(lvl, actions)
            except Exception:
                pass
            return actions
        BFSSolver._scan_actions = scan_wrapped

        orig_solve = BFSSolver.solve_level
        def solve_wrapped(self, level_idx, max_states=500000, prev_solution=None, goal_heuristic=None):
            t0 = time.time()
            try:
                if not hasattr(self, "lister"):
                    self.lister = GameplayLister(prefix="BFS")
                self._glist_active_level = level_idx
                self.lister.emit(level_idx, "SOLVE", f"start max_states={max_states} prev_solution_len={len(prev_solution) if prev_solution else 0}", min_trace=1)
            except Exception:
                pass
            route = None
            try:
                route = orig_solve(self, level_idx, max_states=max_states, prev_solution=prev_solution, goal_heuristic=goal_heuristic)
                return route
            finally:
                try:
                    if route:
                        self.lister.result(level_idx, "solve_level", route, elapsed=time.time() - t0)
                    else:
                        self.lister.result(level_idx, "solve_level", None, elapsed=time.time() - t0, reason="no_solution")
                except Exception:
                    pass
        BFSSolver.solve_level = solve_wrapped

        orig_transfer = BFSSolver._try_transfer
        def transfer_wrapped(self, game, level_idx, prev_solution, f1):
            t0 = time.time()
            try:
                if not hasattr(self, "lister"):
                    self.lister = GameplayLister(prefix="BFS")
                self.lister.route(level_idx, "transfer-input", prev_solution, status="candidate")
            except Exception:
                pass
            route = orig_transfer(self, game, level_idx, prev_solution, f1)
            try:
                if route:
                    self.lister.result(level_idx, "transfer", route, elapsed=time.time() - t0)
                else:
                    self.lister.result(level_idx, "transfer", None, elapsed=time.time() - t0, reason="transfer_failed")
            except Exception:
                pass
            return route
        BFSSolver._try_transfer = transfer_wrapped

        orig_hidden = BFSSolver._probe_hidden_fields
        def hidden_wrapped(self, game, actions):
            fields = orig_hidden(self, game, actions)
            try:
                if hasattr(self, "lister"):
                    self.lister.fields(getattr(self, "_glist_active_level", "?"), "hidden", fields)
            except Exception:
                pass
            return fields
        BFSSolver._probe_hidden_fields = hidden_wrapped

        orig_transient = BFSSolver._detect_transient_fields
        def transient_wrapped(self, game, actions):
            fields = orig_transient(self, game, actions)
            try:
                if hasattr(self, "lister"):
                    self.lister.fields(getattr(self, "_glist_active_level", "?"), "transient", fields)
            except Exception:
                pass
            return fields
        BFSSolver._detect_transient_fields = transient_wrapped

        orig_fb = BFSSolver._movement_fallback_search
        def fallback_wrapped(self, level_idx, max_states=500000, prev_solution=None, time_budget=60):
            t0 = time.time()
            try:
                if not hasattr(self, "lister"):
                    self.lister = GameplayLister(prefix="BFS")
                self.lister.emit(level_idx, "FALLBACK", f"start max_states={max_states} time_budget={time_budget} prev_solution_len={len(prev_solution) if prev_solution else 0}", min_trace=2)
            except Exception:
                pass
            route = orig_fb(self, level_idx, max_states=max_states, prev_solution=prev_solution, time_budget=time_budget)
            try:
                if route:
                    self.lister.result(level_idx, "movement_fallback", route, elapsed=time.time() - t0)
                else:
                    self.lister.result(level_idx, "movement_fallback", None, elapsed=time.time() - t0, reason="no_route")
            except Exception:
                pass
            return route
        BFSSolver._movement_fallback_search = fallback_wrapped

        MyAgent._forge_glist_installed = True
        return True
    except Exception as e:
        try:
            print("[GLIST][INSTALL][ERROR]", type(e).__name__, e, flush=True)
        except Exception:
            pass
        return False


try:
    if _install_inline_gameplay_lister():
        print("[OK] Inline gameplay lister active | env: FORGE_GAMEPLAY_TRACE=1/2/3, FORGE_GAMEPLAY_LOG=0/1", flush=True)
except Exception as e:
    print("[GLIST][INSTALL][ERROR]", type(e).__name__, e, flush=True)

# =====================================================================



In [ ]:
# ============================================================
# 4) Register agent into ARC-AGI-3-Agents repo
# ============================================================
if not REPO_ROOT.exists():
    raise FileNotFoundError(
        f"ARC-AGI-3-Agents repo not found at {REPO_ROOT}. Place arc-prize-2026-arc-agi-3.zip beside this notebook."
    )

TEMPLATE_DIR = REPO_ROOT / "agents" / "templates"
TEMPLATE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copyfile(ROOT / "my_agent.py", TEMPLATE_DIR / "my_agent.py")

# Minimal __init__.py avoids importing optional LLM/template deps that may not exist locally.
init_path = REPO_ROOT / "agents" / "__init__.py"
init_path.write_text('''from typing import Type\nfrom dotenv import load_dotenv\nfrom .agent import Agent, Playback\nfrom .swarm import Swarm\nfrom .templates.random_agent import Random\nfrom .templates.my_agent import MyAgent\nload_dotenv()\nAVAILABLE_AGENTS: dict[str, Type[Agent]] = {"random": Random, "myagent": MyAgent}\n''', encoding="utf-8")

# Ensure imports resolve.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Compile for syntax check before running expensive scorecard logic.
compile_ok = compileall.compile_file(str(TEMPLATE_DIR / "my_agent.py"), quiet=1)
print("[SETUP] registered", TEMPLATE_DIR / "my_agent.py")
print("[SETUP] compile_ok=", compile_ok)
if not compile_ok:
    raise SyntaxError("my_agent.py failed compileall check")


In [ ]:
# ============================================================
# 5) Environment + API key setup
# ============================================================
from getpass import getpass
from arc_agi import Arcade, OperationMode
from arcengine import GameState
from agents.templates.my_agent import MyAgent

os.environ["ENVIRONMENTS_DIR"] = str(ENV_DIR)
os.environ["RECORDINGS_DIR"] = str(RECORDINGS_DIR)
if USE_AGENT_TRAINING and STRATEGY_PRIOR_PATH.exists():
    os.environ["FORGE_STRATEGY_PRIOR_PATH"] = str(STRATEGY_PRIOR_PATH)
os.environ.setdefault("ARC_BASE_URL", "https://three.arcprize.org")

if RUN_TARGET in ("api", "api_competition") and not os.environ.get("ARC_API_KEY"):
    os.environ["ARC_API_KEY"] = getpass("Paste ARC_API_KEY: ").strip()

if RUN_TARGET == "offline":
    OP_MODE = OperationMode.OFFLINE
elif RUN_TARGET == "api_competition":
    OP_MODE = OperationMode.COMPETITION
else:
    OP_MODE = OperationMode.ONLINE

print("[RUN] OP_MODE=", OP_MODE)
print("[RUN] ARC_BASE_URL=", os.environ.get("ARC_BASE_URL"))
print("[RUN] ARC_API_KEY loaded=", bool(os.environ.get("ARC_API_KEY")))
print("[RUN] ENVIRONMENTS_DIR=", os.environ.get("ENVIRONMENTS_DIR"))
print("[RUN] RECORDINGS_DIR=", os.environ.get("RECORDINGS_DIR"))


In [ ]:
# ============================================================
# 6) Direct local/API scorecard runner
#    This bypasses ARC-AGI-3-Agents/main.py so local offline does not need a gateway server.
# ============================================================
import json, time, traceback, logging
from pathlib import Path

logging.basicConfig(
    level=logging.DEBUG if os.environ.get("DEBUG") == "True" else logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(LOG_DIR / "forge_v195_run.log", mode="w")],
    force=True,
)
logger = logging.getLogger("forge_v195_runner")

arc = Arcade(
    arc_api_key=os.environ.get("ARC_API_KEY", ""),
    arc_base_url=os.environ.get("ARC_BASE_URL", "https://three.arcprize.org"),
    operation_mode=OP_MODE,
    environments_dir=str(ENV_DIR),
    recordings_dir=str(RECORDINGS_DIR),
)

envs = arc.get_environments()
print(f"[RUN] discovered {len(envs)} environment entries")

# Use base game IDs once each: ls20 instead of ls20-<version> where applicable.
def base_id(game_id: str) -> str:
    return str(game_id).split("-", 1)[0]

all_games = sorted({base_id(e.game_id) for e in envs if getattr(e, "game_id", None)})
if GAME_FILTER:
    prefixes = [x.strip() for x in GAME_FILTER.split(",") if x.strip()]
    games = [g for g in all_games if any(g.startswith(p) for p in prefixes)]
else:
    games = all_games
if MAX_GAMES > 0:
    games = games[:MAX_GAMES]

print("[RUN] games=", games)
if not games:
    raise RuntimeError("No games selected. Check ARC_API_KEY, RUN_TARGET, and ENVIRONMENTS_DIR.")

ROOT_URL = os.environ.get("ARC_BASE_URL", "https://three.arcprize.org").rstrip("/")


def run_one_game(game_id: str, card_id: str):
    t0 = time.time()
    print(f"\n[GAME][START] {game_id}", flush=True)
    env = arc.make(game_id, scorecard_id=card_id, save_recording=True, include_frame_data=True)
    if env is None:
        print(f"[GAME][ERROR] {game_id}: arc.make returned None", flush=True)
        return {"game_id": game_id, "ok": False, "error": "arc.make returned None"}

    agent = MyAgent(
        card_id=card_id,
        game_id=game_id,
        agent_name="myagent",
        ROOT_URL=ROOT_URL,
        record=True,
        arc_env=env,
        tags=["forge-v19.5", RUN_TARGET, "local-notebook"],
    )
    try:
        agent.main()
        latest = agent.frames[-1] if agent.frames else None
        result = {
            "game_id": game_id,
            "ok": True,
            "actions": int(getattr(agent, "action_counter", -1)),
            "levels_completed": int(getattr(latest, "levels_completed", -1)) if latest is not None else -1,
            "state": str(getattr(latest, "state", "unknown")) if latest is not None else "unknown",
            "seconds": round(time.time() - t0, 3),
        }
        print("[GAME][DONE]", json.dumps(result, default=str), flush=True)
        return result
    except Exception as e:
        traceback.print_exc()
        return {
            "game_id": game_id,
            "ok": False,
            "error": repr(e),
            "seconds": round(time.time() - t0, 3),
        }


def summarize_scorecard(scorecard):
    if scorecard is None:
        return {"scorecard": None}
    try:
        data = scorecard.model_dump()
    except Exception:
        try:
            data = json.loads(scorecard.model_dump_json())
        except Exception:
            data = {"repr": repr(scorecard)}
    return data


In [ ]:
# ============================================================
# 7) Run scorecard
# ============================================================
RUN_TAGS = ["forge-v19.5", "sigilagi", "local-notebook", RUN_TARGET]
card_id = arc.open_scorecard(tags=RUN_TAGS)
print("[SCORECARD][OPEN]", card_id)
if RUN_TARGET in ("api", "api_competition"):
    print("[SCORECARD][URL]", f"{ROOT_URL}/scorecards/{card_id}")

results = []
try:
    for game_id in games:
        results.append(run_one_game(game_id, card_id))
finally:
    print("\n[SCORECARD][CLOSE]", card_id)
    scorecard = arc.close_scorecard(card_id)

scorecard_data = summarize_scorecard(scorecard)

result_path = LOG_DIR / f"forge_v195_results_{int(time.time())}.json"
result_path.write_text(json.dumps({
    "run_target": RUN_TARGET,
    "card_id": card_id,
    "scorecard_url": f"{ROOT_URL}/scorecards/{card_id}" if RUN_TARGET in ("api", "api_competition") else None,
    "games": games,
    "results": results,
    "scorecard": scorecard_data,
}, indent=2, default=str), encoding="utf-8")

print("[RESULTS] wrote", result_path)
print(json.dumps(results, indent=2, default=str))
print("\n[SCORECARD][SUMMARY]")
print(json.dumps(scorecard_data, indent=2, default=str)[:12000])
if RUN_TARGET in ("api", "api_competition"):
    print("\nOpen scorecard:", f"{ROOT_URL}/scorecards/{card_id}")


## 8) Useful reruns

Run one game only:

```bash
RUN_TARGET=api GAME_FILTER=ls20 MAX_GAMES=1 jupyter notebook forge_v195_local_api_scorecard.ipynb
```

Offline local bundle run:

```bash
RUN_TARGET=offline MAX_GAMES=3 jupyter notebook forge_v195_local_api_scorecard.ipynb
```

Heavy lister:

```bash
FORGE_GAMEPLAY_TRACE=3 RUN_TARGET=api GAME_FILTER=g50t jupyter notebook forge_v195_local_api_scorecard.ipynb
```

For Kaggle final submission, keep API calls out of the final notebook and wrap this same `my_agent.py` through the official competition gateway cell.


Training-prior run:

```bash
USE_AGENT_TRAINING=1 RUN_TARGET=api GAME_FILTER=ls20 MAX_GAMES=1 jupyter notebook forge_v195_training_prior_scorecard.ipynb
```

Disable the training prior for an A/B test:

```bash
USE_AGENT_TRAINING=0 RUN_TARGET=api GAME_FILTER=ls20 MAX_GAMES=1 jupyter notebook forge_v195_training_prior_scorecard.ipynb
```
